# NB-02_proceso_ejecucion_gc_cgr

Process Flow SAS: **Proceso Ejecución GC CGR** — `PFD-wyxachgCf8TVObWz`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

Analista = None  # &Analista — nadie declaró su valor
Year = None  # &Year — nadie declaró su valor

faltantes = [n for n, v in {"Analista": Analista, "Year": Year}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text
import pyxlsb

# Conexión a BD — editable acá; SASMIG_DB_URL (orquestador) tiene
# prioridad si está definida (SUPUESTO: verificar servidor y base
# antes de correr contra datos reales).
config_db = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=PLATDAT,1433;"
    "DATABASE=PYDCNIGOB;"
    "Authentication=ActiveDirectoryIntegrated;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "MARS_Connection=Yes;"
)
engine = sqlalchemy.create_engine(
    os.environ.get("SASMIG_DB_URL", f"mssql+pyodbc:///?odbc_connect={config_db}"),
    pool_pre_ping=True,
    fast_executemany=True,
)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-02_proceso_ejecucion_gc_cgr.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


# ── Helper SAS: SOUNDEX (inyectado porque algún nodo lo usa) ──
def _soundex(texto):
    """SOUNDEX de SAS (variante Knuth): largo variable, sin rellenar con ceros.

    Vocales (A E I O U Y) separan códigos repetidos; H y W no codifican ni
    separan. Caracteres no alfabéticos se descartan. None/NaN → "".
    """
    if texto is None or texto != texto:
        return ""
    codigos = {}
    for letras, digito in (("BFPV", "1"), ("CGJKQSXZ", "2"), ("DT", "3"),
                           ("L", "4"), ("MN", "5"), ("R", "6")):
        for letra in letras:
            codigos[letra] = digito
    letras = [c for c in str(texto).upper() if "A" <= c <= "Z"]
    if not letras:
        return ""
    salida = [letras[0]]
    previo = codigos.get(letras[0], "")
    for letra in letras[1:]:
        if letra in "HW":
            continue
        digito = codigos.get(letra, "")
        if digito and digito != previo:
            salida.append(digito)
        previo = digito
    return "".join(salida)


# ── Helper SAS: SPEDIS (inyectado porque algún nodo lo usa) ──
def _spedis(query, keyword):
    """SPEDIS de SAS: distancia de tipeo ASIMÉTRICA de keyword hacia query.

    Replica la tabla de costos de la doc oficial (match 0, singlet 25,
    doublet 50, swap 50, truncate 50, append 35, delete 50, insert 100,
    replace 100, firstdel 100, firstins 200, firstrep 200); el resultado es
    la suma de costos con división ENTERA por el largo de query. Sensible a
    mayúsculas y recorta blancos finales, como el original. None/NaN → None;
    query vacía con keyword no vacía → None (SAS no define ese caso).
    """
    if query is None or keyword is None or query != query or keyword != keyword:
        return None
    q = str(query).rstrip()
    k = str(keyword).rstrip()
    if not q:
        return 0 if not k else None
    inf = float("inf")
    n, m = len(k), len(q)
    dp = [[inf] * (m + 1) for _ in range(n + 1)]
    dp[0][0] = 0
    for i in range(n + 1):
        for j in range(m + 1):
            if i and j:  # match / replace / firstrep
                if k[i - 1] == q[j - 1]:
                    costo = 0
                elif i == 1 and j == 1:
                    costo = 200
                else:
                    costo = 100
                dp[i][j] = min(dp[i][j], dp[i - 1][j - 1] + costo)
            if i:  # borrar k[i-1]: firstdel / singlet / delete-truncate
                if i == 1 and j == 0:
                    costo = 100
                elif i >= 2 and k[i - 1] == k[i - 2]:
                    costo = 25
                else:
                    costo = 50
                dp[i][j] = min(dp[i][j], dp[i - 1][j] + costo)
            if j:  # insertar q[j-1]: firstins / doublet / append / insert
                if j == 1 and i == 0:
                    costo = 200
                else:
                    costo = 100
                    if j >= 2 and q[j - 1] == q[j - 2]:
                        costo = 50
                    if i == n:
                        costo = min(costo, 35)
                dp[i][j] = min(dp[i][j], dp[i][j - 1] + costo)
            if i >= 2 and j >= 2 and k[i - 1] != k[i - 2]                     and k[i - 1] == q[j - 2] and k[i - 2] == q[j - 1]:
                dp[i][j] = min(dp[i][j], dp[i - 2][j - 2] + 50)  # swap
    return int(dp[n][m]) // m


## 1-Check Compara PACAPR

Compara para el año de proceso los códigos y nombres de partida/capítulo/programa de cada entidad contra su versión histórica en ENTIDADES, detectando cambios

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN multi-criterio y SELECT DISTINCT*

In [ ]:
# ========= 1-Check Compara PACAPR =========
work_conn.execute(text("DROP TABLE IF EXISTS #compara_pacapr"))
# año interpolado como entero (no :param) para que la #tmp sobreviva
# REVISA CAMBIOS EN NOMBRES Y CÓDIGOS DE ENTIDADES SEGÚN PACAPR
sql_compara_pacapr = f"""
SELECT DISTINCT t1.C_ENTIDAD,
                t2.C_ENTIDAD AS C_ENTIDAD_ANTIG,
                t1.C_PARTIDA,
                t1.N_PARTIDA,
                t2.N_PARTIDA AS N_PARTIDA_ANTIG,
                t1.C_CAPITULO,
                t1.N_CAPITULO,
                t2.N_CAPITULO AS N_CAPITULO_ANTIG,
                t1.C_PROGRAMA,
                t1.N_PROGRAMA,
                t2.N_PROGRAMA AS N_PROGRAMA_ANTIG
INTO #compara_pacapr
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.ENTIDADES t2
  ON t1.C_PARTIDA = t2.C_PARTIDA
  AND t1.C_CAPITULO = t2.C_CAPITULO
  AND t1.C_PROGRAMA = t2.C_PROGRAMA
WHERE t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_compara_pacapr))


In [ ]:
compara_pacapr = pd.read_sql(text("SELECT * FROM #compara_pacapr"), work_conn)
_log("compara_pacapr", compara_pacapr)


## 2-Check Compara C_ENTIDAD

Compara los códigos y nombres de partida/capítulo/programa vigentes contra los registrados en la tabla de entidades para el año, según código de entidad

*confianza: high · verificador: approve · SAS: PROC SQL CREATE TABLE con SELECT DISTINCT y LEFT JOIN*

In [ ]:
# ========= 2-Check Compara C_ENTIDAD =========
work_conn.execute(text("DROP TABLE IF EXISTS #compara_entidad"))
# REVISA CAMBIOS EN CÓDIGOS DE PACAPR SEGÚN CÓDIGO ENTIDAD
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_compara_entidad = f"""
SELECT DISTINCT t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t2.C_PARTIDA AS C_PARTIDA_ANTIG,
       t1.N_PARTIDA,
       t2.N_PARTIDA AS N_PARTIDA_ANTIG,
       t1.C_CAPITULO,
       t2.C_CAPITULO AS C_CAPITULO_ANTIG,
       t1.N_CAPITULO,
       t2.N_CAPITULO AS N_CAPITULO_ANTIG,
       t1.C_PROGRAMA,
       t2.C_PROGRAMA AS C_PROGRAMA_ANTIG,
       t1.N_PROGRAMA,
       t2.N_PROGRAMA AS N_PROGRAMA_ANTIG
INTO #compara_entidad
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.ENTIDADES t2 ON (t1.C_ENTIDAD = t2.C_ENTIDAD)
WHERE t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_compara_entidad))


In [ ]:
compara_entidad = pd.read_sql(text("SELECT * FROM #compara_entidad"), work_conn)
_log("compara_entidad", compara_entidad)


## 3a-Check Nvas Entidades

Detecta combinaciones de partida/capítulo/programa presentes en la ejecución del año que no existen en el maestro de entidades (entidades nuevas o sin clasificar)

*confianza: high · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN y filtro IS NULL (anti-join)*

In [ ]:
# ========= 3a-Check Nvas Entidades =========
work_conn.execute(text("DROP TABLE IF EXISTS #check_entidades"))
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_check_entidades = f"""
SELECT DISTINCT t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.N_PARTIDA,
       t1.N_CAPITULO,
       t1.N_PROGRAMA
INTO #check_entidades
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.ENTIDADES t2
  ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA)
WHERE (t2.C_ENTIDAD IS NULL) AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_check_entidades))


In [ ]:
check_entidades = pd.read_sql(text("SELECT * FROM #check_entidades"), work_conn)
_log("check_entidades", check_entidades)


## 3b-Clasif Nvas Entidades

Identifica entidades del año presentes en la ejecución presupuestaria que aún no existen en la matriz de entidades y las incorpora a esa matriz

*confianza: high · verificador: approve · SAS: PROC SQL CREATE TABLE DISTINCT con LEFT JOIN (anti-join) + PROC DATASETS APPEND FORCE*

In [ ]:
# ========= 3b-Clasif Nvas Entidades =========
work_conn.execute(text("DROP TABLE IF EXISTS #nuevas_entidades"))
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_nuevas_entidades = f"""
SELECT DISTINCT t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.N_PARTIDA,
       t1.C_CAPITULO,
       t1.N_CAPITULO,
       t1.C_PROGRAMA,
       t1.N_PROGRAMA
INTO #nuevas_entidades
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.ENTIDADES t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
WHERE (t2.C_ENTIDAD IS NULL) AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_nuevas_entidades))
nuevas_entidades = pd.read_sql(text("SELECT * FROM #nuevas_entidades"), work_conn)
_log("nuevas_entidades", nuevas_entidades)


In [ ]:
# ACTUALIZA MATRIZ ENTIDADES CON NUEVAS ENTIDADES DE CARGA
# APPEND server-side: el FORCE de PROC DATASETS alinea por nombre de columna, columnas explícitas
cols_entidades = "C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA"
sql_append_entidades = f"""
INSERT INTO GOBGENER.dbo.ENTIDADES ({cols_entidades})
SELECT {cols_entidades}
FROM #nuevas_entidades
"""
res = work_conn.execute(text(sql_append_entidades))
_log("APPEND GOBGENER.dbo.ENTIDADES", res.rowcount)


## 4a-Check Analiticos Predef_mod

Propone clasificaciones para analíticos fijos del periodo que no están predefinidos, usando la clasificación de mayor frecuencia histórica por clave compuesta (cuenta-ítem-asignación, con clave corta de respaldo), autocompletando el caso conocido cuenta 07-ítem 02 y descartando lo que no pudo clasificarse

*confianza: low · verificador: approve · SAS: PROC SQL: WORK temporal desde tabla de BD + LEFT JOIN de control + join cartesiano con desempate por PROC SORT NODUPKEY + UPDATE/DELETE de depuración*

In [ ]:
# ========= 4a-Check Analiticos Predef_mod =========
# I. VERIFICA ANALITICOS FIJOS FALTANTES. MODIFICADO EN JUNIO DE 2025
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr"))
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_ejec_cgr = f"""
SELECT *
INTO #ejec_cgr
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejec_cgr))


In [ ]:
# DETERMINA ANALITICOS QUE NO ESTÁN EN TABLA FIJA Y QUE REQUIEREN CLASIFICACIÓN
work_conn.execute(text("DROP TABLE IF EXISTS #analit_fijo_faltante"))
sql_analit_fijo_faltante = """
SELECT t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       COUNT(t1.DEVENG) AS ID, t2.C_ANALITICO AS C_ANALIT,
       CONCAT_WS('_', t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION) AS KEY_CORTA,
       CONCAT_WS('_', t1.C_CUENTA, t1.C_ITEM, SUBSTRING(t1.C_ASIGNACION, 1, 2)) AS KEY_CORTA_2
INTO #analit_fijo_faltante
FROM #ejec_cgr t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE (t2.C_ANALITICO IS NULL OR t2.C_ANALITICO = '') AND t1.C_CUENTA NOT IN ('05','13','24','33')
GROUP BY t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t2.C_ANALITICO
"""
work_conn.execute(text(sql_analit_fijo_faltante))
analit_fijo_faltante = pd.read_sql(text("SELECT * FROM #analit_fijo_faltante"), work_conn)
_log("analit_fijo_faltante", analit_fijo_faltante)


In [ ]:
# II. PROPONE CLASIFICACIONES A LOS FALTANTES DEL PERIODO, DEFINIENDO CLAVE Y USANDO LAS CLASIFICACIONES QUE TIENEN MAYOR FRECUENCIA
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, USANDO CUENTA-ITEM-ASIGNACION
# Se agrega _ord (orden de origen materializado) para fijar el orden fisico de GOBGENER.CR18_T_SCN_1,
# que es el criterio que usa PROC SORT NODUPKEY de SAS para desempatar (primer registro por orden fisico)
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_fijo"))
sql_cruce_fijo = """
SELECT CONCAT_WS('_', C_CUENTA, C_ITEM, C_ASIGNACION) AS KEY_CORTA,
       T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION,
       N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID,
       MIN(_rn) AS _ord
INTO #cruce_fijo
FROM (
    SELECT *, ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS _rn
    FROM GOBGENER.dbo.CR18_T_SCN_1
) src
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, T_TRANSACCION
"""
work_conn.execute(text(sql_cruce_fijo))


In [ ]:
# DEFINE LINEA CON MAYOR NUMERO DE REPETICIONES PARA CADA KEY_CORTA
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_fijo_max"))
sql_cruce_fijo_max = """
SELECT KEY_CORTA, MAX(ID) AS ID_MAX
INTO #cruce_fijo_max
FROM #cruce_fijo
GROUP BY KEY_CORTA
"""
work_conn.execute(text(sql_cruce_fijo_max))


In [ ]:
# CLASIFICACION PROPUESTA ANALITICOS FIJOS
# CORRECCION: el join cartesiano de SAS puede generar varias filas por KEY_CORTA cuando hay empate en ID_MAX;
# PROC SORT NODUPKEY posterior se queda con la primera fila segun el orden fisico de origen (_ord).
# Se reemplaza el ORDER BY (SELECT NULL), no determinista, por ORDER BY t1._ord (determinista dentro de la corrida)
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop"))
sql_cruce_prop = """
SELECT KEY_CORTA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION,
       N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, ID
INTO #cruce_prop
FROM (
    SELECT t1.*,
           ROW_NUMBER() OVER (PARTITION BY t1.KEY_CORTA ORDER BY t1._ord) AS _rn
    FROM #cruce_fijo t1
    INNER JOIN #cruce_fijo_max t2 ON t1.KEY_CORTA = t2.KEY_CORTA AND t1.ID = t2.ID_MAX
) d
WHERE _rn = 1
"""
work_conn.execute(text(sql_cruce_prop))
cruce_prop = pd.read_sql(text("SELECT * FROM #cruce_prop"), work_conn)
_log("cruce_prop", cruce_prop)


In [ ]:
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, CON UNA CLAVE MAS CORTA QUE LA ANTERIOR
# (CUENTA-ITEM-ASIGNACION, CON ASIGNACION USANDO LOS DOS PRIMEROS DIGITOS)
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_fijo_2"))
sql_cruce_fijo_2 = """
SELECT CONCAT_WS('_', C_CUENTA, C_ITEM, SUBSTRING(C_ASIGNACION, 1, 2)) AS KEY_CORTA_2,
       C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID,
       MIN(_rn) AS _ord
INTO #cruce_fijo_2
FROM (
    SELECT *, ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS _rn
    FROM GOBGENER.dbo.CR18_T_SCN_1
) src
GROUP BY C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, CONCAT_WS('_', C_CUENTA, C_ITEM, SUBSTRING(C_ASIGNACION, 1, 2))
"""
work_conn.execute(text(sql_cruce_fijo_2))


In [ ]:
# DEFINE LINEA CON MAYOR NUMERO DE REPETICIONES PARA CADA KEY_CORTA_2
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_fijo_max_2"))
sql_cruce_fijo_max_2 = """
SELECT KEY_CORTA_2, MAX(ID) AS ID_MAX
INTO #cruce_fijo_max_2
FROM #cruce_fijo_2
GROUP BY KEY_CORTA_2
"""
work_conn.execute(text(sql_cruce_fijo_max_2))


In [ ]:
# CLASIFICACION PROPUESTA ANALITICOS FIJOS (clave corta 2)
# CORRECCION: mismo criterio de desempate determinista via orden fisico de origen (_ord) que CRUCE_PROP
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop_2"))
sql_cruce_prop_2 = """
SELECT KEY_CORTA_2, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, ID
INTO #cruce_prop_2
FROM (
    SELECT t1.*,
           ROW_NUMBER() OVER (PARTITION BY t1.KEY_CORTA_2 ORDER BY t1._ord) AS _rn
    FROM #cruce_fijo_2 t1
    INNER JOIN #cruce_fijo_max_2 t2 ON t1.KEY_CORTA_2 = t2.KEY_CORTA_2 AND t1.ID = t2.ID_MAX
) d
WHERE _rn = 1
"""
work_conn.execute(text(sql_cruce_prop_2))
cruce_prop_2 = pd.read_sql(text("SELECT * FROM #cruce_prop_2"), work_conn)
_log("cruce_prop_2", cruce_prop_2)


In [ ]:
# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES EN PREDEFINIDOS, USANDO CLAVE 1 (CUENTA-ITEM-ASIGNACION)
# El LEFT JOIN de #analit_fijo_faltante contra #cruce_prop es 1-a-1 por KEY_CORTA (ya deduplicado arriba),
# por lo que no requiere desempate adicional
work_conn.execute(text("DROP TABLE IF EXISTS #analit_fijo_clas"))
# Analista interpolado como literal controlado (viene de macro_params, no de input externo)
sql_analit_fijo_clas = f"""
SELECT t1.C_TIPO_CUENTA AS T_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO AS ANALITICO, t2.C_SCN, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_CAGENTE, t2.C_CUP, t2.C_ACTIVO,
       CONVERT(varchar(9), CAST(GETDATE() AS date), 106) AS FECHA,
       UPPER('{Analista}') AS ANALISTA,
       t1.KEY_CORTA, t1.KEY_CORTA_2
INTO #analit_fijo_clas
FROM #analit_fijo_faltante t1
LEFT JOIN #cruce_prop t2 ON t1.KEY_CORTA = t2.KEY_CORTA
"""
work_conn.execute(text(sql_analit_fijo_clas))


In [ ]:
# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES EN PREDEFINIDOS, USANDO CLAVE CORTA 2
work_conn.execute(text("DROP TABLE IF EXISTS #analit_fijo_clas_2"))
sql_analit_fijo_clas_2 = """
SELECT t1.T_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.ANALITICO,
       t2.C_SCN, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_CAGENTE, t2.C_CUP, t2.C_ACTIVO,
       CONVERT(varchar(9), CAST(GETDATE() AS date), 106) AS FECHA,
       t1.ANALISTA
INTO #analit_fijo_clas_2
FROM #analit_fijo_clas t1
LEFT JOIN #cruce_prop_2 t2 ON t1.KEY_CORTA_2 = t2.KEY_CORTA_2
WHERE (t1.C_SCN IS NULL OR t1.C_SCN = '')
"""
work_conn.execute(text(sql_analit_fijo_clas_2))


In [ ]:
# Deja las clasificaciones faltantes como "", la idea es crearlas a mano
# tabla con clasificaciones propuestas. Revisar y modificar en caso que sea necesario
# CORRECCION: se materializa una columna _seq (ROW_NUMBER sobre el propio UNION ALL, en el orden
# CLAS luego CLAS_2, igual al SET de SAS) para fijar el orden fisico de insercion que el PROC SORT
# NODUPKEY posterior usa como criterio de desempate, en vez de depender de (SELECT NULL)
work_conn.execute(text("DROP TABLE IF EXISTS #analit_fijo_clas_union"))
cols_clas = "T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, ANALITICO, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, FECHA, ANALISTA"
sql_union = f"""
SELECT {cols_clas}, ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS _seq
INTO #analit_fijo_clas_union
FROM (
    SELECT {cols_clas} FROM #analit_fijo_clas
    UNION ALL
    SELECT {cols_clas} FROM #analit_fijo_clas_2
) u
"""
work_conn.execute(text(sql_union))


In [ ]:
# ELIMINA POSIBLES DUPLICADOS (PROC SORT NODUPKEY BY C_CUENTA C_ANALITICO C_ITEM)
# CORRECCION: el SAS no fija un ORDER BY previo a este PROC SORT: NODUPKEY conserva el primer registro
# segun el orden fisico en que llegaron las filas (ANALIT_FIJO_CLAS, luego ANALIT_FIJO_CLAS_2 via SET).
# Ese orden quedo fijado explicitamente en _seq (celda anterior); el desempate usa ORDER BY _seq
# (determinista) en lugar de ORDER BY (SELECT NULL)
work_conn.execute(text("DROP TABLE IF EXISTS #analit_fijo_clas_final"))
sql_dedup = f"""
SELECT {cols_clas}
INTO #analit_fijo_clas_final
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY C_CUENTA, C_ANALITICO, C_ITEM ORDER BY _seq) AS _rn
    FROM #analit_fijo_clas_union
) d
WHERE _rn = 1
"""
work_conn.execute(text(sql_dedup))


In [ ]:
# En general las cuentas 07-02 se asocian al p11, por lo que autocompleta con este código
res = work_conn.execute(text("""
UPDATE #analit_fijo_clas_final
SET C_SCN = 'P11', C_TRANSACCION = '111', T_TRANSACCION = 'Prod merc', N_CUENTA = 'Ventas', C_CAGENTE = 'S1'
WHERE (C_SCN IS NULL OR C_SCN = '') AND C_CUENTA = '07' AND C_ITEM = '02'
"""))
_log("UPDATE #analit_fijo_clas_final autocompleta 07-02", res.rowcount)


In [ ]:
# Borra los registros cuando no se pueden clasificar
res = work_conn.execute(text("DELETE FROM #analit_fijo_clas_final WHERE ANALITICO = 'SIN DENOMINACION'"))
_log("DELETE #analit_fijo_clas_final ANALITICO=SIN DENOMINACION", res.rowcount)
analit_fijo_clas = pd.read_sql(text("SELECT * FROM #analit_fijo_clas_final"), work_conn)
_log("analit_fijo_clas", analit_fijo_clas)
for t in ["#ejec_cgr", "#analit_fijo_faltante", "#cruce_fijo", "#cruce_fijo_max", "#cruce_prop", "#cruce_fijo_2", "#cruce_fijo_max_2", "#cruce_prop_2", "#analit_fijo_clas", "#analit_fijo_clas_2", "#analit_fijo_clas_union", "#analit_fijo_clas_final"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## 4b-Clasif Analiticos Predef_mod

Reemplaza en CR18_T_SCN_1 las clasificaciones ya cubiertas por ANALIT_FIJO_CLAS, marca como 'REV' las sin clasificación asignada, anexa el resultado a la base fija y deja tablas de control de duplicados y pendientes de revisión

*confianza: medium · verificador: approve · SAS: PROC SQL DELETE con EXISTS-subquery + UPDATE + PROC DATASETS APPEND + verificaciones de duplicados/pendientes*

In [ ]:
# ========= 4b-Clasif Analiticos Predef_mod =========
# I. ANEXA CLASIFICACIONES FALTANTES DESDE PASO ANTERIOR
# ANALIT_FIJO_CLAS queda materializado como #tmp_analit_fijo_clas en la sesión (work_conn) para poder correlacionar el DELETE en el servidor
work_conn.execute(text("DROP TABLE IF EXISTS #tmp_analit_fijo_clas"))
analit_fijo_clas.to_sql("#tmp_analit_fijo_clas", work_conn, if_exists="replace", index=False)

res = work_conn.execute(text("""
DELETE FROM GOBGENER.dbo.CR18_T_SCN_1
WHERE EXISTS (
    SELECT 1 FROM #tmp_analit_fijo_clas t
    WHERE t.T_CUENTA = CR18_T_SCN_1.T_CUENTA
      AND t.C_CUENTA = CR18_T_SCN_1.C_CUENTA
      AND t.C_ITEM = CR18_T_SCN_1.C_ITEM
      AND t.C_ASIGNACION = CR18_T_SCN_1.C_ASIGNACION
      AND t.C_ANALITICO = CR18_T_SCN_1.C_ANALITICO
)
"""))
_log("DELETE GOBGENER.dbo.CR18_T_SCN_1", res.rowcount)


In [ ]:
# UPDATE ANALIT_FIJO_CLAS SET C_SCN='REV' WHERE C_SCN='' (comodín para revisión manual posterior)
work_conn.execute(text("""
UPDATE #tmp_analit_fijo_clas
SET C_SCN = 'REV'
WHERE C_SCN IS NULL OR C_SCN = ''
"""))
analit_fijo_clas = pd.read_sql(text("SELECT * FROM #tmp_analit_fijo_clas"), work_conn)
_log("analit_fijo_clas", analit_fijo_clas)


In [ ]:
# APPEND BASE=GOBGENER.CR18_T_SCN_1 DATA=ANALIT_FIJO_CLAS FORCE
analit_fijo_clas.to_sql("CR18_T_SCN_1", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# VERIFICA QUE NO EXISTAN DUPLICADOS EN LA BASE DE CLASIFICACIONES FIJAS (SI HAY REPETIDOS MOSTRARÁ CAMPOS EN ESTA SALIDA)
sql_check_analit_fijo = """
SELECT T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, COUNT(C_SCN) AS ID
FROM GOBGENER.dbo.CR18_T_SCN_1
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO
HAVING COUNT(C_SCN) <> 1
"""
check_analit_fijo = pd.read_sql(text(sql_check_analit_fijo), engine)
_log("check_analit_fijo", check_analit_fijo)


In [ ]:
# VERIFICA QUE NO EXISTAN ANALITICOS SIN CLASIFICAR
sql_check_analit_rev = """
SELECT T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, C_SCN
FROM GOBGENER.dbo.CR18_T_SCN_1
WHERE C_SCN = 'REV'
"""
check_analit_rev = pd.read_sql(text(sql_check_analit_rev), engine)
_log("check_analit_rev", check_analit_rev)


## 5a-Check Analiticos Var_mod

Detecta analíticos variables sin clasificación del periodo, propone clasificación heredando el registro más frecuente entre 5 niveles de clave decrecientes (GORE primero, luego resto de entidades), marca observaciones especiales por tipo de transacción y exporta la propuesta a Excel para revisión manual

*confianza: medium · verificador: approve · SAS: PROC SQL con múltiples niveles de LEFT JOIN sobre claves fallback (KEY_1..KEY_5) + PROC SORT NODUPKEY + PROC DATASETS APPEND + PROC EXPORT a Excel*

In [ ]:
# ========= 5a-Check Analiticos Var_mod =========
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr"))
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_ejec_cgr = f"""
SELECT *
INTO #ejec_cgr
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejec_cgr))


In [ ]:
# SELECCIONA ANALITICOS VARIABLES SIN ENTIDAD, QUE QUEDAN ASÍ AL CARGAR DIPRES, PARA INCORPORARLES LA ENTIDAD
work_conn.execute(text("DROP TABLE IF EXISTS #ent_n"))
sql_ent_n = """
SELECT DISTINCT C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_ENTIDAD
INTO #ent_n
FROM #ejec_cgr
"""
work_conn.execute(text(sql_ent_n))


In [ ]:
# incorpora entidad para actualizar datos
work_conn.execute(text("DROP TABLE IF EXISTS #analit_sin_entidad"))
sql_analit_sin_entidad = """
SELECT T1.*, T2.C_ENTIDAD
INTO #analit_sin_entidad
FROM GOBGENER.dbo.CR18_T_SCN_2 T1
LEFT JOIN #ent_n T2 ON T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
WHERE T1.ENTIDAD IS NULL
"""
work_conn.execute(text(sql_analit_sin_entidad))


In [ ]:
work_conn.execute(text("UPDATE #analit_sin_entidad SET ENTIDAD = C_ENTIDAD"))
# DATA step DROP=C_ENTIDAD: la columna auxiliar se descarta al armar el INSERT del APPEND, no altera la #tmp


In [ ]:
# los elimina para agregarlos con la entidad
res = work_conn.execute(text("DELETE FROM GOBGENER.dbo.CR18_T_SCN_2 WHERE ENTIDAD IS NULL"))
_log("DELETE GOBGENER.dbo.CR18_T_SCN_2 (sin entidad)", res.rowcount)


In [ ]:
# PROC APPEND FORCE: alinea por nombre, columna auxiliar C_ENTIDAD queda excluida
cols_cr18_t_scn_2 = pd.read_sql(text("SELECT TOP 0 * FROM GOBGENER.dbo.CR18_T_SCN_2"), work_conn).columns.tolist()
cols_append = ", ".join(cols_cr18_t_scn_2)
sql_append_analit_sin_entidad = f"""
INSERT INTO GOBGENER.dbo.CR18_T_SCN_2 ({cols_append})
SELECT {cols_append}
FROM #analit_sin_entidad
"""
res = work_conn.execute(text(sql_append_analit_sin_entidad))
_log("APPEND GOBGENER.dbo.CR18_T_SCN_2", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ent_n"))


In [ ]:
# MODIFICADO EN JUNIO 2025
# I. DETERMINA ANALITICOS FALTANTES VARIABLES POR CLASIFICAR DEL PERIODO
# ANALITICOS QUE NO ESTÁN EN TABLA DE CLASIFICACIONES
work_conn.execute(text("DROP TABLE IF EXISTS #analit_entidad_var"))
sql_analit_entidad_var = """
SELECT T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO, COUNT(T1.DEVENG) AS ID, T2.C_ANALITICO AS C_ANALIT
INTO #analit_entidad_var
FROM #ejec_cgr T1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 T2 ON T1.C_ENTIDAD = T2.ENTIDAD AND T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA AND T1.C_TIPO_CUENTA = T2.T_CUENTA AND T1.C_CUENTA = T2.C_CUENTA AND T1.C_ANALITICO = T2.C_ANALITICO
WHERE (T2.C_ANALITICO IS NULL OR T2.C_ANALITICO = '') AND T1.C_CUENTA IN ('05','13','24','33')
GROUP BY T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO, T2.C_ANALITICO
"""
work_conn.execute(text(sql_analit_entidad_var))


In [ ]:
# II. PROPONE CLASIFICACIONES PARA GORE (PARTIDA=88)
work_conn.execute(text("ALTER TABLE #analit_entidad_var ADD KEY_1 varchar(200), KEY_2 varchar(200)"))
work_conn.execute(text("UPDATE #analit_entidad_var SET KEY_1 = CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ITEM, C_ASIGNACION), KEY_2 = CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ASIGNACION)"))


In [ ]:
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, USANDO KEY_1
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var"))
sql_cruce_var = """
SELECT CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ITEM, C_ASIGNACION) AS KEY_1, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID
INTO #cruce_var
FROM GOBGENER.dbo.CR18_T_SCN_2
WHERE C_PARTIDA = '88'
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, T_TRANSACCION, C_PARTIDA
"""
work_conn.execute(text(sql_cruce_var))
cruce_var = pd.read_sql(text("SELECT * FROM #cruce_var"), work_conn)
_log("cruce_var", cruce_var)


In [ ]:
# DEFINE LINEA CON MAYOR NUMERO DE REPETICIONES PARA CADA KEY_CORTA
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_max"))
sql_cruce_var_max = """
SELECT KEY_1, MAX(ID) AS ID_MAX, COUNT(ID) AS ID
INTO #cruce_var_max
FROM #cruce_var
GROUP BY KEY_1
"""
work_conn.execute(text(sql_cruce_var_max))


In [ ]:
# CLASIFICACION PROPUESTA ANALITICOS FIJOS
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop"))
sql_cruce_prop = """
SELECT T1.*, T2.ID_MAX, T2.ID AS ID_COUNT
INTO #cruce_prop
FROM #cruce_var T1, #cruce_var_max T2
WHERE T1.KEY_1 = T2.KEY_1 AND T1.ID = T2.ID_MAX
"""
work_conn.execute(text(sql_cruce_prop))


In [ ]:
# ELIMINA POSIBLES DUPLICADOS (PROC SORT NODUPKEY BY KEY_1)
cruce_prop = pd.read_sql(text("SELECT * FROM #cruce_prop"), work_conn)
cruce_prop = cruce_prop.sort_values("KEY_1").drop_duplicates(subset=["KEY_1"], keep="first")
cruce_prop.to_sql("#cruce_prop", work_conn, if_exists="replace", index=False)
_log("cruce_prop", cruce_prop)


In [ ]:
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, USANDO KEY_2
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_2"))
sql_cruce_var_2 = """
SELECT CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ASIGNACION) AS KEY_2, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID
INTO #cruce_var_2
FROM GOBGENER.dbo.CR18_T_SCN_2
WHERE C_PARTIDA = '88'
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, T_TRANSACCION, C_PARTIDA
"""
work_conn.execute(text(sql_cruce_var_2))
cruce_var_2 = pd.read_sql(text("SELECT * FROM #cruce_var_2"), work_conn)
_log("cruce_var_2", cruce_var_2)


In [ ]:
# DEFINE LINEA CON MAYOR NUMERO DE REPETICIONES PARA CADA KEY_CORTA
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_max_2"))
sql_cruce_var_max_2 = """
SELECT KEY_2, MAX(ID) AS ID_MAX, COUNT(ID) AS ID
INTO #cruce_var_max_2
FROM #cruce_var_2
GROUP BY KEY_2
"""
work_conn.execute(text(sql_cruce_var_max_2))


In [ ]:
# CLASIFICACION PROPUESTA ANALITICOS FIJOS, KEY_2
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop_2"))
sql_cruce_prop_2 = """
SELECT T1.*, T2.ID_MAX, T2.ID AS ID_COUNT
INTO #cruce_prop_2
FROM #cruce_var_2 T1, #cruce_var_max_2 T2
WHERE T1.KEY_2 = T2.KEY_2 AND T1.ID = T2.ID_MAX
"""
work_conn.execute(text(sql_cruce_prop_2))


In [ ]:
# ELIMINA POSIBLES DUPLICADOS (PROC SORT NODUPKEY BY KEY_2)
cruce_prop_2 = pd.read_sql(text("SELECT * FROM #cruce_prop_2"), work_conn)
cruce_prop_2 = cruce_prop_2.sort_values("KEY_2").drop_duplicates(subset=["KEY_2"], keep="first")
cruce_prop_2.to_sql("#cruce_prop_2", work_conn, if_exists="replace", index=False)
_log("cruce_prop_2", cruce_prop_2)


In [ ]:
# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES GORES, USANDO KEY_1
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas"))
# analista interpolado como literal controlado (macro_params) para que la #tmp sobreviva
sql_analit_var_clas = f"""
SELECT T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA AS T_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO AS ANALITICO, T2.C_SCN, T2.C_TRANSACCION, T2.T_TRANSACCION, T2.N_CUENTA, T2.C_CAGENTE, T2.C_CUP, T2.C_ACTIVO, T1.C_ENTIDAD AS ENTIDAD, CAST(GETDATE() AS date) AS FECHA, UPPER('{Analista}') AS ANALISTA, T1.KEY_1, T1.KEY_2, T2.ID_COUNT
INTO #analit_var_clas
FROM #analit_entidad_var T1
LEFT JOIN #cruce_prop T2 ON T1.KEY_1 = T2.KEY_1
WHERE T1.C_PARTIDA = '88'
"""
work_conn.execute(text(sql_analit_var_clas))


In [ ]:
# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES GORES, USANDO KEY_2
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_2"))
sql_analit_var_clas_2 = f"""
SELECT T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.T_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.ANALITICO, T2.C_SCN, T2.C_TRANSACCION, T2.T_TRANSACCION, T2.N_CUENTA, T2.C_CAGENTE, T2.C_CUP, T2.C_ACTIVO, T1.ENTIDAD, CAST(GETDATE() AS date) AS FECHA, UPPER('{Analista}') AS ANALISTA, T2.ID_COUNT
INTO #analit_var_clas_2
FROM #analit_var_clas T1
LEFT JOIN #cruce_prop_2 T2 ON T1.KEY_2 = T2.KEY_2
WHERE (T1.C_SCN IS NULL OR T1.C_SCN = '')
"""
work_conn.execute(text(sql_analit_var_clas_2))


In [ ]:
res = work_conn.execute(text("DELETE FROM #analit_var_clas WHERE C_SCN IS NULL OR C_SCN = ''"))
_log("DELETE #analit_var_clas (sin clasificar tras KEY_1)", res.rowcount)


In [ ]:
# UNION de ambas propuestas GORE, sin las columnas de clave auxiliares
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_gore"))
cols_gore = "C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, ANALITICO, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, ENTIDAD, FECHA, ANALISTA"
sql_analit_var_clas_gore = f"""
SELECT {cols_gore}
INTO #analit_var_clas_gore
FROM #analit_var_clas
UNION ALL
SELECT {cols_gore}
FROM #analit_var_clas_2
"""
work_conn.execute(text(sql_analit_var_clas_gore))
analit_var_clas_gore = pd.read_sql(text("SELECT * FROM #analit_var_clas_gore"), work_conn)
_log("analit_var_clas_gore", analit_var_clas_gore)


In [ ]:
for t in ["#cruce_var", "#cruce_var_max", "#cruce_var_2", "#cruce_var_max_2", "#analit_var_clas_2", "#cruce_prop", "#cruce_prop_2", "#analit_var_clas"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# III. PROPONE CLASIFICACIONES PARA RESTO DE ENTIDADES (DISTINTOS DE GORE)
work_conn.execute(text("ALTER TABLE #analit_entidad_var DROP COLUMN KEY_1, KEY_2"))
work_conn.execute(text("ALTER TABLE #analit_entidad_var ADD KEY_3 varchar(200), KEY_4 varchar(200)"))
work_conn.execute(text("UPDATE #analit_entidad_var SET KEY_3 = CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ITEM, C_ASIGNACION), KEY_4 = CONCAT_WS('_', C_PARTIDA, C_PROGRAMA, C_CUENTA, C_ASIGNACION)"))


In [ ]:
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, USANDO KEY_3
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_3"))
sql_cruce_var_3 = """
SELECT CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ITEM, C_ASIGNACION) AS KEY_3, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID
INTO #cruce_var_3
FROM GOBGENER.dbo.CR18_T_SCN_2
WHERE C_PARTIDA NOT IN ('88')
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, T_TRANSACCION, C_PARTIDA
"""
work_conn.execute(text(sql_cruce_var_3))
cruce_var_3 = pd.read_sql(text("SELECT * FROM #cruce_var_3"), work_conn)
_log("cruce_var_3", cruce_var_3)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_max_3"))
sql_cruce_var_max_3 = """
SELECT KEY_3, MAX(ID) AS ID_MAX, COUNT(ID) AS ID
INTO #cruce_var_max_3
FROM #cruce_var_3
GROUP BY KEY_3
"""
work_conn.execute(text(sql_cruce_var_max_3))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop_3"))
sql_cruce_prop_3 = """
SELECT T1.*, T2.ID_MAX, T2.ID AS ID_COUNT
INTO #cruce_prop_3
FROM #cruce_var_3 T1, #cruce_var_max_3 T2
WHERE T1.KEY_3 = T2.KEY_3 AND T1.ID = T2.ID_MAX
"""
work_conn.execute(text(sql_cruce_prop_3))


In [ ]:
# ELIMINA POSIBLES DUPLICADOS (PROC SORT NODUPKEY BY KEY_3)
cruce_prop_3 = pd.read_sql(text("SELECT * FROM #cruce_prop_3"), work_conn)
cruce_prop_3 = cruce_prop_3.sort_values("KEY_3").drop_duplicates(subset=["KEY_3"], keep="first")
cruce_prop_3.to_sql("#cruce_prop_3", work_conn, if_exists="replace", index=False)
_log("cruce_prop_3", cruce_prop_3)


In [ ]:
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, USANDO KEY_4
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_4"))
sql_cruce_var_4 = """
SELECT CONCAT_WS('_', C_PARTIDA, C_PROGRAMA, C_CUENTA, C_ASIGNACION) AS KEY_4, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID
INTO #cruce_var_4
FROM GOBGENER.dbo.CR18_T_SCN_2
WHERE C_PARTIDA NOT IN ('88')
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, T_TRANSACCION, C_PARTIDA, C_PROGRAMA
"""
work_conn.execute(text(sql_cruce_var_4))
cruce_var_4 = pd.read_sql(text("SELECT * FROM #cruce_var_4"), work_conn)
_log("cruce_var_4", cruce_var_4)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_max_4"))
sql_cruce_var_max_4 = """
SELECT KEY_4, MAX(ID) AS ID_MAX, COUNT(ID) AS ID
INTO #cruce_var_max_4
FROM #cruce_var_4
GROUP BY KEY_4
"""
work_conn.execute(text(sql_cruce_var_max_4))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop_4"))
sql_cruce_prop_4 = """
SELECT T1.*, T2.ID_MAX, T2.ID AS ID_COUNT
INTO #cruce_prop_4
FROM #cruce_var_4 T1, #cruce_var_max_4 T2
WHERE T1.KEY_4 = T2.KEY_4 AND T1.ID = T2.ID_MAX
"""
work_conn.execute(text(sql_cruce_prop_4))


In [ ]:
# ELIMINA POSIBLES DUPLICADOS (PROC SORT NODUPKEY BY KEY_4)
cruce_prop_4 = pd.read_sql(text("SELECT * FROM #cruce_prop_4"), work_conn)
cruce_prop_4 = cruce_prop_4.sort_values("KEY_4").drop_duplicates(subset=["KEY_4"], keep="first")
cruce_prop_4.to_sql("#cruce_prop_4", work_conn, if_exists="replace", index=False)
_log("cruce_prop_4", cruce_prop_4)


In [ ]:
# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES, USANDO KEY_3
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_3"))
sql_analit_var_clas_3 = f"""
SELECT T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA AS T_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO AS ANALITICO, T2.C_SCN, T2.C_TRANSACCION, T2.T_TRANSACCION, T2.N_CUENTA, T2.C_CAGENTE, T2.C_CUP, T2.C_ACTIVO, T1.C_ENTIDAD AS ENTIDAD, CAST(GETDATE() AS date) AS FECHA, UPPER('{Analista}') AS ANALISTA, T1.KEY_3, T1.KEY_4, T2.ID_COUNT
INTO #analit_var_clas_3
FROM #analit_entidad_var T1
LEFT JOIN #cruce_prop_3 T2 ON T1.KEY_3 = T2.KEY_3
WHERE T1.C_PARTIDA NOT IN ('88')
"""
work_conn.execute(text(sql_analit_var_clas_3))


In [ ]:
# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES, USANDO KEY_4
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_4"))
sql_analit_var_clas_4 = f"""
SELECT T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.T_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.ANALITICO, T2.C_SCN, T2.C_TRANSACCION, T2.T_TRANSACCION, T2.N_CUENTA, T2.C_CAGENTE, T2.C_CUP, T2.C_ACTIVO, T1.ENTIDAD, CAST(GETDATE() AS date) AS FECHA, UPPER('{Analista}') AS ANALISTA, T1.KEY_3, T1.KEY_4, T2.ID_COUNT
INTO #analit_var_clas_4
FROM #analit_var_clas_3 T1
LEFT JOIN #cruce_prop_4 T2 ON T1.KEY_4 = T2.KEY_4
WHERE T1.C_PARTIDA NOT IN ('88') AND (T1.C_SCN IS NULL OR T1.C_SCN = '')
"""
work_conn.execute(text(sql_analit_var_clas_4))


In [ ]:
res = work_conn.execute(text("DELETE FROM #analit_var_clas_3 WHERE C_SCN IS NULL OR C_SCN = ''"))
_log("DELETE #analit_var_clas_3 (sin clasificar tras KEY_3)", res.rowcount)


In [ ]:
# UNION de ambas propuestas RESTO, sin las columnas de clave auxiliares
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_resto"))
cols_resto = "C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, ANALITICO, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, ENTIDAD, FECHA, ANALISTA"
sql_analit_var_clas_resto = f"""
SELECT {cols_resto}
INTO #analit_var_clas_resto
FROM #analit_var_clas_3
UNION ALL
SELECT {cols_resto}
FROM #analit_var_clas_4
"""
work_conn.execute(text(sql_analit_var_clas_resto))
analit_var_clas_resto = pd.read_sql(text("SELECT * FROM #analit_var_clas_resto"), work_conn)
_log("analit_var_clas_resto", analit_var_clas_resto)


In [ ]:
for t in ["#cruce_var_3", "#cruce_var_max_3", "#cruce_var_4", "#cruce_var_max_4", "#analit_var_clas_4", "#cruce_prop_3", "#cruce_prop_4", "#analit_var_clas_3"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IV. PROPONE CLASIFICACIONES PARA RESTO DE ENTIDADES (DISTINTOS DE GORE), FALTANTES DE PASO ANTERIOR, EN CASO QUE SE MANTENGA LA ENTIDAD PERO CAMBIE LA PARTIDA
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_resto_f"))
sql_analit_var_clas_resto_f = """
SELECT *, CONCAT_WS('_', ENTIDAD, C_CUENTA, C_ITEM, C_ASIGNACION) AS KEY_5
INTO #analit_var_clas_resto_f
FROM #analit_var_clas_resto
WHERE C_SCN IS NULL OR C_SCN = ''
"""
work_conn.execute(text(sql_analit_var_clas_resto_f))
analit_var_clas_resto_f = pd.read_sql(text("SELECT * FROM #analit_var_clas_resto_f"), work_conn)
_log("analit_var_clas_resto_f", analit_var_clas_resto_f)


In [ ]:
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, USANDO KEY_5
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_5"))
sql_cruce_var_5 = """
SELECT CONCAT_WS('_', ENTIDAD, C_CUENTA, C_ITEM, C_ASIGNACION) AS KEY_5, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID
INTO #cruce_var_5
FROM GOBGENER.dbo.CR18_T_SCN_2
WHERE C_PARTIDA NOT IN ('88')
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, T_TRANSACCION, ENTIDAD
"""
work_conn.execute(text(sql_cruce_var_5))
cruce_var_5 = pd.read_sql(text("SELECT * FROM #cruce_var_5"), work_conn)
_log("cruce_var_5", cruce_var_5)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_max_5"))
sql_cruce_var_max_5 = """
SELECT KEY_5, MAX(ID) AS ID_MAX, COUNT(ID) AS ID
INTO #cruce_var_max_5
FROM #cruce_var_5
GROUP BY KEY_5
"""
work_conn.execute(text(sql_cruce_var_max_5))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop_5"))
sql_cruce_prop_5 = """
SELECT T1.*, T2.ID_MAX, T2.ID AS ID_COUNT
INTO #cruce_prop_5
FROM #cruce_var_5 T1, #cruce_var_max_5 T2
WHERE T1.KEY_5 = T2.KEY_5 AND T1.ID = T2.ID_MAX
"""
work_conn.execute(text(sql_cruce_prop_5))


In [ ]:
# ELIMINA POSIBLES DUPLICADOS (PROC SORT NODUPKEY BY KEY_5)
cruce_prop_5 = pd.read_sql(text("SELECT * FROM #cruce_prop_5"), work_conn)
cruce_prop_5 = cruce_prop_5.sort_values("KEY_5").drop_duplicates(subset=["KEY_5"], keep="first")
cruce_prop_5.to_sql("#cruce_prop_5", work_conn, if_exists="replace", index=False)
_log("cruce_prop_5", cruce_prop_5)


In [ ]:
# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES, USANDO KEY_5
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_5"))
sql_analit_var_clas_5 = f"""
SELECT T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.T_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.ANALITICO, T2.C_SCN, T2.C_TRANSACCION, T2.T_TRANSACCION, T2.N_CUENTA, T2.C_CAGENTE, T2.C_CUP, T2.C_ACTIVO, T1.ENTIDAD, CAST(GETDATE() AS date) AS FECHA, UPPER('{Analista}') AS ANALISTA, T2.ID_COUNT
INTO #analit_var_clas_5
FROM #analit_var_clas_resto_f T1
LEFT JOIN #cruce_prop_5 T2 ON T1.KEY_5 = T2.KEY_5
"""
work_conn.execute(text(sql_analit_var_clas_5))
analit_var_clas_5 = pd.read_sql(text("SELECT * FROM #analit_var_clas_5"), work_conn)
_log("analit_var_clas_5", analit_var_clas_5)


In [ ]:
for t in ["#cruce_var_5", "#cruce_var_max_5", "#cruce_prop_5"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# UNE TABLAS FINALES DE CLASIFICACIONES PROPUESTAS: solo las que quedaron con C_SCN asignado en RESTO
res = work_conn.execute(text("DELETE FROM #analit_var_clas_resto WHERE C_SCN IS NULL OR C_SCN = ''"))
_log("DELETE #analit_var_clas_resto (sin clasificar)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #check_transvar"))
cols_final = "C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, ANALITICO, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, ENTIDAD, FECHA, ANALISTA"
sql_check_transvar = f"""
SELECT {cols_final}, 'Ingresar texto si aplica' AS OBS
INTO #check_transvar
FROM #analit_var_clas_gore
UNION ALL
SELECT {cols_final}, 'Ingresar texto si aplica' AS OBS
FROM #analit_var_clas_resto
UNION ALL
SELECT {cols_final}, 'Ingresar texto si aplica' AS OBS
FROM #analit_var_clas_5
"""
work_conn.execute(text(sql_check_transvar))


In [ ]:
res = work_conn.execute(text("UPDATE #check_transvar SET OBS = 'CR18_difcoy', C_ACTIVO = 'D63' WHERE C_SCN IN ('D63-P2','D63-P')"))
_log("UPDATE #check_transvar OBS=CR18_difcoy", res.rowcount)
res = work_conn.execute(text("UPDATE #check_transvar SET OBS = 'CR18_difact' WHERE C_SCN = 'P3'"))
_log("UPDATE #check_transvar OBS=CR18_difact (P3)", res.rowcount)
res = work_conn.execute(text("UPDATE #check_transvar SET OBS = 'CR18_difact' WHERE C_SCN = 'P2' AND C_CUP IN ('EEPP','SSPP')"))
_log("UPDATE #check_transvar OBS=CR18_difact (P2)", res.rowcount)
check_transvar = pd.read_sql(text("SELECT * FROM #check_transvar"), work_conn)
_log("check_transvar", check_transvar)


In [ ]:
for t in ["#analit_var_clas_gore", "#analit_var_clas_resto", "#analit_var_clas_5", "#analit_var_clas_resto_f", "#ejec_cgr"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# lleva propuesta de clasificaciones a excel para su revisión, completar los faltantes y luego en paso siguiente cargar a SAS
# ruta UNC original reemplazada por ruta relativa al workspace; PROC EXPORT DBMS=EXCELCS es el bridge PC Files Server: se lee/escribe directo por archivo, sin servidor intermedio ni credenciales
ruta_check_transvar = Path("salidas") / "CHECK_TRANSVAR_PROP.xlsb"
check_transvar.to_excel(ruta_check_transvar, sheet_name="PROPUESTA", index=False, engine="pyxlsb")


## 5b-Clasif Analiticos Var_mod

Importa la propuesta de clasificación de analíticos variables desde Excel, normaliza observaciones y activo, descarta filas ya clasificadas en la tabla oficial, anexa las nuevas y verifica que no queden claves duplicadas

*confianza: medium · verificador: approve · SAS: PROC IMPORT EXCELCS + PROC SQL DELETE/UPDATE + anti-join DELETE + PROC DATASETS APPEND + PROC SQL GROUP BY control de duplicados*

In [ ]:
# ========= 5b-Clasif Analiticos Var_mod =========
# MODIFICADO JUNIO 2025
# IMPORTA CLASIFICACIONES
# DBMS=EXCELCS (PC Files Server): en Python se lee el archivo directo por su ruta; no hay servidor intermedio ni credenciales que replicar.
# Ruta original UNC del SAS reemplazada por ruta relativa al workspace (ver warnings).
ruta_check_transvar_p = Path("insumos") / "GobiernoGeneral" / "CHECK_TRANSVAR_PROP.xlsb"
check_transvar_p = pd.read_excel(ruta_check_transvar_p, sheet_name="PROPUESTA", engine="pyxlsb")
_log("check_transvar_p", check_transvar_p)


In [ ]:
# Elimina filas totalmente sin clasificación (blancos en las 4 columnas clave)
check_transvar_p = check_transvar_p[
    ~(
        (check_transvar_p["C_PARTIDA"].isna() | (check_transvar_p["C_PARTIDA"] == ""))
        & (check_transvar_p["C_CAPITULO"].isna() | (check_transvar_p["C_CAPITULO"] == ""))
        & (check_transvar_p["C_PROGRAMA"].isna() | (check_transvar_p["C_PROGRAMA"] == ""))
        & (check_transvar_p["C_ANALITICO"].isna() | (check_transvar_p["C_ANALITICO"] == ""))
    )
].copy()

# ALTER TABLE ... MODIFY C_ACTIVO CHAR(6): trunca/normaliza el ancho de la columna a 6 caracteres
check_transvar_p["C_ACTIVO"] = check_transvar_p["C_ACTIVO"].astype(str).str.slice(0, 6)
check_transvar_p.loc[check_transvar_p["C_ACTIVO"] == "nan", "C_ACTIVO"] = ""

# UPDATE: limpia el texto de instructivo por defecto
check_transvar_p.loc[check_transvar_p["OBS"] == "Ingresar texto si aplica", "OBS"] = ""

# UPDATE: limpia OBS para C_SCN fuera de la lista de interés, salvo que ya esté vacío
check_transvar_p.loc[
    (~check_transvar_p["C_SCN"].isin(["D63-P2", "D63-P", "P3", "P2"]))
    & (check_transvar_p["OBS"] != ""),
    "OBS",
] = ""

# UPDATE: marca CR18_difcoy/C_ACTIVO='D63' para C_SCN D63-P2/D63-P sin OBS
mask_d63 = check_transvar_p["C_SCN"].isin(["D63-P2", "D63-P"]) & (check_transvar_p["OBS"] == "")
check_transvar_p.loc[mask_d63, "OBS"] = "CR18_difcoy"
check_transvar_p.loc[mask_d63, "C_ACTIVO"] = "D63"

# UPDATE: marca CR18_difact para C_SCN='P3' sin OBS
mask_p3 = (check_transvar_p["C_SCN"] == "P3") & (check_transvar_p["OBS"] == "")
check_transvar_p.loc[mask_p3, "OBS"] = "CR18_difact"

# UPDATE: marca CR18_difact para C_SCN='P2' con C_CUP en EEPP/SSPP sin OBS
mask_p2 = (
    (check_transvar_p["C_SCN"] == "P2")
    & (check_transvar_p["C_CUP"].isin(["EEPP", "SSPP"]))
    & (check_transvar_p["OBS"] == "")
)
check_transvar_p.loc[mask_p2, "OBS"] = "CR18_difact"
_log("check_transvar_p", check_transvar_p)


In [ ]:
# Sube el resultado de pandas a una tabla temporal de sesión para operar el resto en SQL
work_conn.execute(text("DROP TABLE IF EXISTS #check_transvar_p"))
check_transvar_p.to_sql("#check_transvar_p", work_conn, if_exists="replace", index=False)


In [ ]:
# ANEXA CLASIFICACIONES FALTANTES
# Elimina de la temporal las filas que ya existen (misma clave) en GOBGENER.CR18_T_SCN_2
work_conn.execute(text("""
DELETE FROM #check_transvar_p
WHERE EXISTS (
    SELECT 1 FROM GOBGENER.dbo.CR18_T_SCN_2 t2
    WHERE t2.C_PARTIDA = #check_transvar_p.C_PARTIDA
      AND t2.C_CAPITULO = #check_transvar_p.C_CAPITULO
      AND t2.C_PROGRAMA = #check_transvar_p.C_PROGRAMA
      AND t2.T_CUENTA = #check_transvar_p.T_CUENTA
      AND t2.C_CUENTA = #check_transvar_p.C_CUENTA
      AND t2.C_ITEM = #check_transvar_p.C_ITEM
      AND t2.C_ASIGNACION = #check_transvar_p.C_ASIGNACION
      AND t2.C_ANALITICO = #check_transvar_p.C_ANALITICO
      AND t2.ENTIDAD = #check_transvar_p.ENTIDAD
)
"""))


In [ ]:
# PROC DATASETS APPEND: acumula lo que sobrevivió del filtro anti-join en GOBGENER.CR18_T_SCN_2
check_transvar_p_final = pd.read_sql(text("SELECT * FROM #check_transvar_p"), work_conn)
check_transvar_p_final.to_sql("CR18_T_SCN_2", engine, schema="dbo", if_exists="append", index=False)
_log("check_transvar_p_final (append CR18_T_SCN_2)", check_transvar_p_final)


In [ ]:
# VERIFICA QUE NO EXISTAN DUPLICADOS EN LA BASE DE CLASIFICACIONES VARIABLES
# SI HAY REPETIDOS MOSTRARÁ CAMPOS EN ESTA SALIDA
sql_check_analit_var = """
SELECT ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, COUNT(C_SCN) AS ID
FROM GOBGENER.dbo.CR18_T_SCN_2
GROUP BY ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO
HAVING COUNT(C_SCN) <> 1
"""
check_analit_var = pd.read_sql(text(sql_check_analit_var), engine)
_log("check_analit_var", check_analit_var)


## 6a-Check Nvas TCons_mod

Detecta transferencias del año sin clasificación previa de Consolidable/No Consolidable dentro de las cuentas e ítems de transferencias, descarta los reversos de ítem 08 no TC y marca como OK el resto fuera del sector TC, exportando el resultado a Excel

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con doble LEFT JOIN + DELETE + UPDATE sobre WORK temporal, y PROC EXPORT a Excel*

In [ ]:
# ========= 6a-Check Nvas TCons_mod =========
work_conn.execute(text("DROP TABLE IF EXISTS #check_tcons"))
# REVISA SI HAY NUEVAS TRANSFERENCIAS CLASIFICABLES EN LOS TIPOS DE CONSOLIDABLES-NOCONSOLIDABLES (TCons)
# MODIFICADO JUNIO 2025
# CHECK COMPARADO CON ACTUAL MATRIZ DE ANALITICOS CLASIFICADOS COMO TCons
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_check_tcons = f"""
SELECT DISTINCT t1.C_CUENTA,
                t1.C_ITEM,
                t1.C_ASIGNACION,
                t1.C_ANALITICO,
                t1.N_ANALITICO AS ANALITICO,
                t1.C_ENTIDAD AS ENTIDAD,
                t2.Clasif_Tcons,
                t2.M_Muestra,
                t3.C_SCN,
                t3.C_CAGENTE,
                t1.C_PARTIDA,
                t1.N_PARTIDA,
                t1.C_CAPITULO,
                t1.N_CAPITULO,
                t1.C_PROGRAMA,
                t1.N_PROGRAMA,
                t3.FECHA,
                t3.ANALISTA
INTO #check_tcons
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.ANALIT_TCONS t2 ON (t1.C_ENTIDAD = t2.ENTIDAD) AND (t2.C_ASIGNACION = t1.C_ASIGNACION)
    AND (t2.C_ANALITICO = t1.C_ANALITICO) AND (t2.C_CUENTA = t1.C_CUENTA) AND (t2.C_ITEM = t1.C_ITEM)
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t3 ON (t1.C_PARTIDA = t3.C_PARTIDA) AND (t1.C_CAPITULO = t3.C_CAPITULO)
    AND (t1.C_PROGRAMA = t3.C_PROGRAMA) AND (t1.C_ASIGNACION = t3.C_ASIGNACION) AND
    (t1.C_ANALITICO = t3.C_ANALITICO) AND (t1.C_CUENTA = t3.C_CUENTA) AND (t1.C_ITEM = t3.C_ITEM) AND (t1.C_ENTIDAD = t3.ENTIDAD)
WHERE t2.Clasif_Tcons IS NULL AND t1.AÑO = {int(Year)} AND t1.C_CUENTA IN ('24','33') AND t1.C_ITEM IN ('01','03','06','09','08')
"""
work_conn.execute(text(sql_check_tcons))


In [ ]:
res = work_conn.execute(text("DELETE FROM #check_tcons WHERE C_ITEM = :c_item AND C_SCN <> :c_scn"), {"c_item": "08", "c_scn": "TC"})
_log("DELETE #check_tcons", res.rowcount)


In [ ]:
res = work_conn.execute(text("UPDATE #check_tcons SET Clasif_Tcons = :clasif WHERE C_SCN NOT IN (:c_scn)"), {"clasif": "OK", "c_scn": "TC"})
_log("UPDATE #check_tcons", res.rowcount)


In [ ]:
check_tcons = pd.read_sql(text("SELECT * FROM #check_tcons"), work_conn)
_log("check_tcons", check_tcons)


In [ ]:
# PROC EXPORT DBMS=EXCELCS: bridge PC Files Server de SAS; en Python se escribe directo a la ruta local/relativa
salida_path = Path("salidas") / "CHECK_TCONS.xlsb"
check_tcons.to_excel(salida_path, sheet_name="TCONS", index=False)


## 6b-Clasif Nvas TCons_mod

Importa clasificaciones TCONS desde Excel, normaliza y marca M_MUESTRA, aísla inconsistencias, elimina y reemplaza registros sin clasificación en ANALIT_TCONS anexando los nuevos, y reporta faltantes de clasificación TCONS en CR18_T_SCN_2

*confianza: medium · verificador: approve · SAS: PROC IMPORT EXCELCS + PROC SQL UPDATE/DELETE/CREATE + PROC DATASETS APPEND*

In [ ]:
# ========= 6b-Clasif Nvas TCons_mod =========
# IMPORTA CLASIFICACIONES
# DBMS=EXCELCS es el bridge PC Files Server de SAS: en Python se lee directo por su ruta (UNC original reemplazada por ruta relativa al workspace)
ruta_check_tcons = Path("insumos") / "CHECK_TCONS.xlsb"
check_tcons_p = pd.read_excel(ruta_check_tcons, sheet_name="TCONS", engine="pyxlsb")
_log("check_tcons_p", check_tcons_p)


In [ ]:
check_tcons_p["CLASIF_TCONS"] = check_tcons_p["CLASIF_TCONS"].str.upper()
check_tcons_p.loc[check_tcons_p["CLASIF_TCONS"].isin(["M", "MM", "MGCG", "MGG"]), "M_MUESTRA"] = "0"


In [ ]:
# DEFINE REGISTROS A CARGAR CON PROBLEMAS DE CONSISTENCIA EN SU LLENADO
check_tcons_r = check_tcons_p[
    (~check_tcons_p["CLASIF_TCONS"].isin(["M", "MM", "MGCG", "MGG", "OK"]))
    & (check_tcons_p["C_CUENTA"].notna() & (check_tcons_p["C_CUENTA"] != ""))
]
_log("check_tcons_r", check_tcons_r)


In [ ]:
# Sube CHECK_TCONS_P a la sesión SQL para operar server-side
check_tcons_p.to_sql("#check_tcons_p", work_conn, if_exists="replace", index=False)


In [ ]:
# ELIMINA T_CONS QUE QUEDARON ANTERIORMENTE SIN CLASIFICACION PARA QUE LA AGREGUE CLASIFICADAS A LA TABLA
res = work_conn.execute(text("DELETE FROM GOBGENER.dbo.ANALIT_TCONS WHERE COALESCE(CLASIF_TCONS, '') = ''"))
_log("DELETE GOBGENER.dbo.ANALIT_TCONS", res.rowcount)


In [ ]:
# ANEXA CLASIFICACIONES FALTANTES: descarta de CHECK_TCONS_P lo que ya existe en ANALIT_TCONS
sql_delete_existentes = """
DELETE FROM #check_tcons_p
WHERE EXISTS (
    SELECT 1 FROM GOBGENER.dbo.ANALIT_TCONS a
    WHERE a.C_CUENTA = #check_tcons_p.C_CUENTA
      AND a.C_ITEM = #check_tcons_p.C_ITEM
      AND a.C_ASIGNACION = #check_tcons_p.C_ASIGNACION
      AND a.C_ANALITICO = #check_tcons_p.C_ANALITICO
      AND a.ENTIDAD = #check_tcons_p.ENTIDAD
)
"""
res = work_conn.execute(text(sql_delete_existentes))
_log("DELETE #check_tcons_p (ya existentes en ANALIT_TCONS)", res.rowcount)


In [ ]:
# Refresca el dataframe en memoria con lo que sobrevivió al DELETE (registros nuevos a anexar)
check_tcons_p = pd.read_sql(text("SELECT * FROM #check_tcons_p"), work_conn)
_log("check_tcons_p (a anexar)", check_tcons_p)


In [ ]:
# PROC APPEND: acumula tal cual, re-ejecutar duplica igual que el SAS original
check_tcons_p.to_sql("ANALIT_TCONS", engine, schema="dbo", if_exists="append", index=False)


In [ ]:
# Verifica consistencia tcons en tabla 2 de clasificaciones: FALTANTES EN TABLA ANALIT_TCONS
sql_check_tcons_falt = """
SELECT T1.ENTIDAD, T1.C_CUENTA, T1.C_ITEM, T1.C_ANALITICO, T1.ANALITICO, T1.C_SCN, T1.C_CAGENTE, T1.FECHA, T1.ANALISTA, T2.CLASIF_TCONS
FROM GOBGENER.dbo.CR18_T_SCN_2 T1
LEFT JOIN GOBGENER.dbo.ANALIT_TCONS T2 ON (T1.ENTIDAD = T2.ENTIDAD AND T1.C_CUENTA = T2.C_CUENTA AND T1.C_ITEM = T2.C_ITEM AND T1.C_ANALITICO = T2.C_ANALITICO)
WHERE T1.C_SCN = 'TC' AND (T2.CLASIF_TCONS IS NULL OR T2.CLASIF_TCONS = '')
ORDER BY T1.FECHA
"""
check_tcons_falt = pd.read_sql(text(sql_check_tcons_falt), engine)
_log("check_tcons_falt", check_tcons_falt)


## 7a-Check Est_Exp_MM TCons_mod

Detecta analíticos MM sin estructura mensual por entidad, exporta a Excel para clasificación manual, valida que las ponderaciones importadas sumen 1 por mes/cuenta y anexa las nuevas estructuras válidas a EST_EXP_MM

*confianza: medium · verificador: approve · SAS: PROC SQL con LEFT JOIN + DATA steps de expansión mensual + PROC EXPORT/IMPORT vía PC Files Server + PROC SQL de validación + PROC DATASETS APPEND*

In [ ]:
# ========= 7a-Check Est_Exp_MM TCons_mod =========
work_conn.execute(text("DROP TABLE IF EXISTS #check_est_exp_mm"))
sql_check_est_exp_mm = """
SELECT DISTINCT t1.C_CUENTA, t1.C_ITEM, t1.ENTIDAD, t1.C_ASIGNACION, t1.C_ANALITICO, t1.ANALITICO,
    '' AS C_SCN, CAST(NULL AS float) AS POND, '' AS C_CAGENTE
INTO #check_est_exp_mm
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN GOBGENER.dbo.EST_EXP_MM t2
    ON t1.ENTIDAD = t2.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
    AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
WHERE t1.Clasif_Tcons IN ('MM') AND t1.M_Muestra IN (0)
    AND (t2.ENTIDAD IS NULL OR t2.ENTIDAD = '')
    AND (t2.C_ANALITICO IS NULL OR t2.C_ANALITICO = '')
"""
work_conn.execute(text(sql_check_est_exp_mm))


In [ ]:
# trae a memoria para las transformaciones fila a fila (N_FALTANTE, 12 meses, MES)
check_est_exp_mm = pd.read_sql(text("SELECT * FROM #check_est_exp_mm"), work_conn)


In [ ]:
# genera campo que define las observaciones faltantes
check_est_exp_mm = check_est_exp_mm.reset_index(drop=True)
n = np.arange(1, len(check_est_exp_mm) + 1)
check_est_exp_mm["N_FALTANTE"] = np.mod(1, n) + 1


In [ ]:
# REPITE CADA FILA 12 VECES, PARA PODER GENERAR ESTRUCTURA DE LOS MESES DEL AÑO
check_est_exp_mm = check_est_exp_mm.loc[check_est_exp_mm.index.repeat(12)].reset_index(drop=True)


In [ ]:
# genera datos para campo meses para que se llene automáticamente
n = np.arange(1, len(check_est_exp_mm) + 1)
check_est_exp_mm["MES"] = np.mod(n - 1, 12) + 1
_log("check_est_exp_mm", check_est_exp_mm)


In [ ]:
# módulo 1: exporta datos a excel para clasificación manual
# ruta original UNC vía PC Files Server (EXCELCS): se lee/escribe directo por la ruta del archivo
ruta_check_est_exp_mm = Path("salidas") / "CHECK_EST_EXP_MM.xlsb"
check_est_exp_mm.to_excel(ruta_check_est_exp_mm, sheet_name="CHECK_EST_EXP_MM", index=False)


In [ ]:
# módulo 2: importa clasificaciones desde excel y las ingresa a la tabla principal
est_exp_mm_c = pd.read_excel(ruta_check_est_exp_mm, sheet_name="CHECK_EST_EXP_MM", engine="pyxlsb")
est_exp_mm_c.to_sql("#est_exp_mm_c", work_conn, if_exists="replace", index=False)


In [ ]:
work_conn.execute(text("DELETE FROM #est_exp_mm_c WHERE (C_CUENTA IS NULL OR C_CUENTA = '') AND (C_ITEM IS NULL OR C_ITEM = '')"))


In [ ]:
# VERIFICA QUE ESTRUCTURAS INGRESADAS TENGAN CONSISTENCIA, ES DECIR SUMEN 1
work_conn.execute(text("DROP TABLE IF EXISTS #check_mm_total_c"))
sql_check_mm_total_c = """
SELECT MES, C_CUENTA, C_ITEM, ENTIDAD, C_ASIGNACION, C_ANALITICO, SUM(POND) AS TOTAL
INTO #check_mm_total_c
FROM #est_exp_mm_c
GROUP BY MES, C_CUENTA, C_ITEM, ENTIDAD, C_ASIGNACION, C_ANALITICO
"""
work_conn.execute(text(sql_check_mm_total_c))
work_conn.execute(text("DELETE FROM #check_mm_total_c WHERE TOTAL = 1"))
check_mm_total_c = pd.read_sql(text("SELECT * FROM #check_mm_total_c"), work_conn)
_log("check_mm_total_c", check_mm_total_c)


In [ ]:
# CHEQUEA CONSISTENCIA ANALITICO PARA DETECTAR POSIBLES ERRORES EN CÓDIGOS, SI SALE VACÍA ESTA OK
sql_check_mm_analit = """
SELECT MES, C_CUENTA, C_ITEM, ENTIDAD, C_ANALITICO,
    CONCAT(C_ITEM, C_ASIGNACION) AS ANALIT,
    CASE WHEN SUBSTRING(C_ANALITICO, 1, LEN(C_ANALITICO) - 3) <> CONCAT(C_ITEM, C_ASIGNACION) THEN 'REV' ELSE 'OK' END AS ID,
    SUM(POND) AS TOTAL
FROM #est_exp_mm_c
WHERE SUBSTRING(C_ANALITICO, 1, LEN(C_ANALITICO) - 3) <> CONCAT(C_ITEM, C_ASIGNACION)
GROUP BY MES, C_CUENTA, C_ITEM, ENTIDAD, C_ANALITICO
"""
check_mm_analit = pd.read_sql(text(sql_check_mm_analit), work_conn)
_log("check_mm_analit", check_mm_analit)


In [ ]:
# ANEXA DATOS A TABLA PRINCIPAL: descarta columnas auxiliares antes de comparar/anexar
work_conn.execute(text("ALTER TABLE #est_exp_mm_c DROP COLUMN ANALITICO"))
work_conn.execute(text("ALTER TABLE #est_exp_mm_c DROP COLUMN N_FALTANTE"))


In [ ]:
sql_delete_existentes = """
DELETE FROM #est_exp_mm_c
WHERE EXISTS (
    SELECT 1 FROM GOBGENER.dbo.EST_EXP_MM e
    WHERE e.C_CUENTA = #est_exp_mm_c.C_CUENTA AND e.C_ITEM = #est_exp_mm_c.C_ITEM
        AND e.C_ASIGNACION = #est_exp_mm_c.C_ASIGNACION AND e.C_ANALITICO = #est_exp_mm_c.C_ANALITICO
        AND e.ENTIDAD = #est_exp_mm_c.ENTIDAD
)
"""
work_conn.execute(text(sql_delete_existentes))


In [ ]:
# PROC APPEND: acumula en la tabla permanente sin borrar lo existente (re-ejecutar duplica, igual que el SAS original)
sql_append = """
INSERT INTO GOBGENER.dbo.EST_EXP_MM (C_CUENTA, C_ITEM, ENTIDAD, C_ASIGNACION, C_ANALITICO, C_SCN, POND, C_CAGENTE, MES)
SELECT C_CUENTA, C_ITEM, ENTIDAD, C_ASIGNACION, C_ANALITICO, C_SCN, POND, C_CAGENTE, MES
FROM #est_exp_mm_c
"""
res = work_conn.execute(text(sql_append))
_log("APPEND GOBGENER.dbo.EST_EXP_MM", res.rowcount)


## 7b-Check Est_Exp_M TCons_mod

Detecta analíticos MIXTAS_M sin estructura expandida por entidad, exporta a Excel para clasificación manual, valida que las ponderaciones ingresadas sumen 1 y que el código analítico sea consistente, y anexa las estructuras nuevas a la tabla oficial evitando duplicados

*confianza: medium · verificador: approve · SAS: PROC SQL LEFT JOIN + DATA steps de expansión mensual + PROC EXPORT/IMPORT EXCELCS + validaciones SQL + PROC DATASETS APPEND*

In [ ]:
# ========= 7b-Check Est_Exp_M TCons_mod =========
# CHECK CONSISTENCIA DE ACTUALIZACION DE ESTRUCTURA EXPANDIDA M (EST_EXP_MIXTAS_M)
# CONTENGA NUEVAS TCONS CLASIFICADAS COMO MIXTAS_M

# módulo 1: determina faltantes y exporta datos a excel
# DETECTA ANALÍTICOS SIN ESTRUCTURAS POR ENTIDAD
work_conn.execute(text("DROP TABLE IF EXISTS #check_est_exp_m"))
sql_check_est_exp_m = """
SELECT DISTINCT t1.C_CUENTA, t1.C_ITEM, t1.ENTIDAD, t1.C_ASIGNACION, t1.C_ANALITICO, t1.ANALITICO,
    '' AS C_SCN, CAST(NULL AS float) AS POND, '' AS C_CAGENTE, '' AS C_CUP
INTO #check_est_exp_m
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN GOBGENER.dbo.EST_EXP_MIXTAS_M t2
    ON t1.ENTIDAD = t2.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
    AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
WHERE t1.Clasif_Tcons IN ('M') AND t1.M_Muestra IN (0)
    AND (t2.ENTIDAD IS NULL) AND (t2.C_ANALITICO IS NULL)
"""
work_conn.execute(text(sql_check_est_exp_m))


In [ ]:
# genera campo que define las observaciones faltantes
check_est_exp_m = pd.read_sql(text("SELECT * FROM #check_est_exp_m"), work_conn)
check_est_exp_m["N_FALTANTE"] = np.arange(1, len(check_est_exp_m) + 1)


In [ ]:
# REPITE CADA FILA 12 VECES, PARA PODER GENERAR ESTRUCTURA DE LOS MESES DEL AÑO
check_est_exp_m = check_est_exp_m.loc[check_est_exp_m.index.repeat(12)].reset_index(drop=True)


In [ ]:
# genera datos para campo meses para que se llene automáticamente
check_est_exp_m["MES"] = (np.arange(len(check_est_exp_m)) % 12) + 1
_log("check_est_exp_m", check_est_exp_m)


In [ ]:
# módulo 1: exporta datos a excel (PROC EXPORT DBMS=EXCELCS -> archivo directo por su ruta UNC; sin servidor intermedio)
ruta_check_est_exp_m = Path("salidas") / "CHECK_EST_EXP_M.xlsb"
check_est_exp_m.to_excel(ruta_check_est_exp_m, sheet_name="CHECK_EST_EXP_M", index=False)


In [ ]:
# módulo 2: importa clasificaciones desde excel y las ingresa a la tabla principal
# IMPORTA CLASIFICACIONES (PROC IMPORT DBMS=EXCELCS -> lectura directa del archivo, sin servidor intermedio)
est_exp_m_c = pd.read_excel(ruta_check_est_exp_m, sheet_name="CHECK_EST_EXP_M")
est_exp_m_c = est_exp_m_c[~((est_exp_m_c["C_CUENTA"].fillna("") == "") & (est_exp_m_c["C_ITEM"].fillna("") == ""))]


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #est_exp_m_c"))
est_exp_m_c.to_sql("#est_exp_m_c", work_conn, if_exists="replace", index=False)


In [ ]:
# VERIFICA QUE ESTRUCTURAS INGRESADAS TENGAN CONSISTENCIA, ES DECIR SUMEN 1
work_conn.execute(text("DROP TABLE IF EXISTS #check_mixtas_tot"))
sql_check_mixtas_tot = """
SELECT MES, C_CUENTA, C_ITEM, ENTIDAD, C_ASIGNACION, C_ANALITICO,
    SUM(POND) AS TOTAL, ROUND(SUM(POND), 1) AS TOTAL2
INTO #check_mixtas_tot
FROM #est_exp_m_c
GROUP BY MES, C_CUENTA, C_ITEM, ENTIDAD, C_ASIGNACION, C_ANALITICO
"""
work_conn.execute(text(sql_check_mixtas_tot))
work_conn.execute(text("DELETE FROM #check_mixtas_tot WHERE ROUND(TOTAL2, 0) = 1"))
check_mixtas_tot = pd.read_sql(text("SELECT * FROM #check_mixtas_tot"), work_conn)
_log("check_mixtas_tot", check_mixtas_tot)


In [ ]:
# CHEQUEA CONSISTENCIA ANALITICO PARA DETECTAR POSIBLES ERRORES EN CÓDIGOS, SI SALE VACÍA ESTA OK
work_conn.execute(text("DROP TABLE IF EXISTS #check_mixtas_analit"))
sql_check_mixtas_analit = """
SELECT C_CUENTA, C_ITEM, ENTIDAD, C_ANALITICO,
    CONCAT(C_ITEM, C_ASIGNACION) AS ANALIT,
    CASE WHEN SUBSTRING(C_ANALITICO, 1, LEN(C_ANALITICO) - 3) <> CONCAT(C_ITEM, C_ASIGNACION) THEN 'REV' ELSE 'OK' END AS ID,
    SUM(POND) AS TOTAL
INTO #check_mixtas_analit
FROM #est_exp_m_c
WHERE SUBSTRING(C_ANALITICO, 1, LEN(C_ANALITICO) - 3) <> CONCAT(C_ITEM, C_ASIGNACION)
GROUP BY C_CUENTA, C_ITEM, ENTIDAD, C_ANALITICO, CONCAT(C_ITEM, C_ASIGNACION),
    CASE WHEN SUBSTRING(C_ANALITICO, 1, LEN(C_ANALITICO) - 3) <> CONCAT(C_ITEM, C_ASIGNACION) THEN 'REV' ELSE 'OK' END
"""
work_conn.execute(text(sql_check_mixtas_analit))


In [ ]:
# CRUCE MALOS PERO NO GENERAN PROBLEMAS (excepciones hardcodeadas del SAS original)
work_conn.execute(text("UPDATE #check_mixtas_analit SET ID = 'OK2' WHERE C_ANALITICO = :c_analitico AND ANALIT = :analit AND ENTIDAD = :entidad"), {"c_analitico": "03027000", "analit": "01027000", "entidad": 1301})
work_conn.execute(text("UPDATE #check_mixtas_analit SET ID = 'OK2' WHERE C_ANALITICO = :c_analitico AND ANALIT = :analit AND ENTIDAD = :entidad"), {"c_analitico": "03515000", "analit": "01515000", "entidad": 6104})
work_conn.execute(text("UPDATE #check_mixtas_analit SET ID = 'OK2' WHERE C_ANALITICO = :c_analitico AND ANALIT = :analit AND ENTIDAD = :entidad"), {"c_analitico": "03604000", "analit": "01604000", "entidad": 6104})
work_conn.execute(text("UPDATE #check_mixtas_analit SET ID = 'OK2' WHERE C_ANALITICO = :c_analitico AND ANALIT = :analit AND ENTIDAD = :entidad"), {"c_analitico": "01114000", "analit": "01004000", "entidad": 24404})
work_conn.execute(text("UPDATE #check_mixtas_analit SET ID = 'OK2' WHERE C_ANALITICO = :c_analitico AND ANALIT = :analit AND ENTIDAD = :entidad"), {"c_analitico": "09010000", "analit": "09007000", "entidad": 26706})
work_conn.execute(text("DELETE FROM #check_mixtas_analit WHERE ID = 'OK2'"))
check_mixtas_analit = pd.read_sql(text("SELECT * FROM #check_mixtas_analit"), work_conn)
_log("check_mixtas_analit", check_mixtas_analit)


In [ ]:
# ANEXA DATOS A TABLA PRINCIPAL
work_conn.execute(text("ALTER TABLE #est_exp_m_c DROP COLUMN ANALITICO, N_FALTANTE"))


In [ ]:
# elimina de EST_EXP_M_C las filas que ya existen en GOBGENER.EST_EXP_MIXTAS_M (anti-join)
sql_delete_existentes = """
DELETE FROM #est_exp_m_c
WHERE EXISTS (
    SELECT 1 FROM GOBGENER.dbo.EST_EXP_MIXTAS_M m
    WHERE m.C_CUENTA = #est_exp_m_c.C_CUENTA AND m.C_ITEM = #est_exp_m_c.C_ITEM
        AND m.C_ASIGNACION = #est_exp_m_c.C_ASIGNACION AND m.C_ANALITICO = #est_exp_m_c.C_ANALITICO
        AND m.ENTIDAD = #est_exp_m_c.ENTIDAD
)
"""
work_conn.execute(text(sql_delete_existentes))


In [ ]:
# PROC DATASETS APPEND BASE=GOBGENER.EST_EXP_MIXTAS_M DATA=EST_EXP_M_C FORCE -> acumula tal cual (re-ejecutar duplica, igual que el SAS original)
est_exp_m_c_final = pd.read_sql(text("SELECT * FROM #est_exp_m_c"), work_conn)
est_exp_m_c_final.to_sql("EST_EXP_MIXTAS_M", engine, schema="dbo", if_exists="append", index=False)
_log("EST_EXP_MIXTAS_M append", len(est_exp_m_c_final))


## 7c-Check Est_MM TCons

Detecta TCons clasificadas como MM y provenientes de la muestra que faltan en la estructura expandida MM (EST_EXP_MM); en estricto rigor no debieran existir

*confianza: high · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN y filtro de no-coincidencia*

In [ ]:
# ========= 7c-Check Est_MM TCons =========
work_conn.execute(text("DROP TABLE IF EXISTS #check_est_mm"))
# CHECK CONSISTENCIA DE ACTUALIZACION DE ESTRUCTURA EXPANDIDA MM (EST_EXP_MM)
# CONTENGA NUEVAS TCONS CLASIFICADAS COMO MM
sql_check_est_mm = """
SELECT DISTINCT t1.*
INTO #check_est_mm
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN GOBGENER.dbo.EST_MM t2
  ON (t1.ENTIDAD = t2.ENTIDAD) AND (t2.C_ASIGNACION = t1.C_ASIGNACION)
  AND (t2.C_ANALITICO = t1.C_ANALITICO) AND (t2.C_CUENTA = t1.C_CUENTA) AND (t2.C_ITEM = t1.C_ITEM)
WHERE t1.Clasif_Tcons = 'MM' AND t1.M_Muestra = 1
  AND (t2.ENTIDAD IS NULL) AND (t2.C_ANALITICO IS NULL)
"""
work_conn.execute(text(sql_check_est_mm))


In [ ]:
check_est_mm = pd.read_sql(text("SELECT * FROM #check_est_mm"), work_conn)
_log("check_est_mm", check_est_mm)


## 7d-Check Est_M TCons

Detecta TCONS clasificadas como Mixtas_M y pertenecientes a la muestra que no están registradas en la estructura expandida EST_MIXTAS_M

*confianza: high · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN anti-match (control de consistencia)*

In [ ]:
# ========= 7d-Check Est_M TCons =========
work_conn.execute(text("DROP TABLE IF EXISTS #check_est_m"))
# CHECK CONSISTENCIA DE ACTUALIZACION DE ESTRUCTURA EXPANDIDA M (EST_EXP_MIXTAS_M)
# CONTENGA NUEVAS TCONS CLASIFICADAS COMO MIXTAS_M
sql_check_est_m = """
SELECT DISTINCT t1.*
INTO #check_est_m
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN GOBGENER.dbo.EST_MIXTAS_M t2
    ON t1.ENTIDAD = t2.ENTIDAD
    AND t2.C_ASIGNACION = t1.C_ASIGNACION
    AND t2.C_ANALITICO = t1.C_ANALITICO
    AND t2.C_CUENTA = t1.C_CUENTA
    AND t2.C_ITEM = t1.C_ITEM
WHERE t1.Clasif_Tcons = 'M' AND t1.M_Muestra = 1
    AND (t2.ENTIDAD IS NULL OR t2.ENTIDAD = '')
    AND (t2.C_ANALITICO IS NULL OR t2.C_ANALITICO = '')
"""
work_conn.execute(text(sql_check_est_m))


In [ ]:
# EN ESTRICTO RIGOR NO DEBIERAN APARECER TCONS MM PROVENIENTES DE LA MUESTRA (1) QUE FALTEN,
# PORQUE LAS NUEVAS SON FUERA DE LA MUESTRA, ES DECIR (0)
check_est_m = pd.read_sql(text("SELECT * FROM #check_est_m"), work_conn)
_log("check_est_m", check_est_m)


## 7e-Check Est_Exp_MGCG TCons_mod

Detecta combinaciones cuenta-item de transferencias consolidables MGCG sin estructura definida, propone una estructura tomando el ítem '03' como plantilla, permite su revisión/edición manual en Excel y la incorpora a la tabla de estructuras evitando duplicados

*confianza: medium · verificador: approve · SAS: PROC SQL (detección de faltantes + propuesta) + PROC EXPORT/IMPORT vía bridge Excel + PROC SQL DELETE/APPEND (PROC DATASETS)*

In [ ]:
# ========= 7e-Check Est_Exp_MGCG TCons_mod =========
# módulo 1: determina faltantes y exporta datos a excel
# DETECTA cuentas-item SIN ESTRUCTURAS
work_conn.execute(text("DROP TABLE IF EXISTS #check_est_exp_mgcg"))
sql_check_est_exp_mgcg = """
SELECT DISTINCT t1.C_CUENTA, t1.C_ITEM, COUNT(t1.C_ASIGNACION) AS ID
INTO #check_est_exp_mgcg
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN GOBGENER.dbo.EST_EXP_MGCG t2 ON (t2.C_CUENTA = t1.C_CUENTA) AND (t2.C_ITEM = t1.C_ITEM)
WHERE t1.Clasif_Tcons = 'MGCG' AND t1.M_Muestra = 0 AND t2.C_SCN IS NULL
GROUP BY t1.C_CUENTA, t1.C_ITEM
"""
work_conn.execute(text(sql_check_est_exp_mgcg))
check_est_exp_mgcg = pd.read_sql(text("SELECT * FROM #check_est_exp_mgcg"), work_conn)
_log("check_est_exp_mgcg", check_est_exp_mgcg)


In [ ]:
# Propone estructura si es que apareciera un item nuevo para la cuenta 24 clasificado en este tipo de transferencia consolidable
work_conn.execute(text("DROP TABLE IF EXISTS #check_est_exp_mgcg_p"))
sql_check_est_exp_mgcg_p = """
SELECT t1.C_CUENTA, t1.C_ITEM, t2.C_SCN, t2.POND, t2.C_CAGENTE, t2.MES
INTO #check_est_exp_mgcg_p
FROM #check_est_exp_mgcg t1
LEFT JOIN GOBGENER.dbo.EST_EXP_MGCG t2 ON (t2.C_CUENTA = t1.C_CUENTA)
WHERE t2.C_ITEM = '03'
"""
work_conn.execute(text(sql_check_est_exp_mgcg_p))
check_est_exp_mgcg_p = pd.read_sql(text("SELECT * FROM #check_est_exp_mgcg_p"), work_conn)
_log("check_est_exp_mgcg_p", check_est_exp_mgcg_p)


In [ ]:
# módulo 1 (cont.): exporta la propuesta a Excel — ruta original de red reemplazada por ruta relativa al workspace
ruta_check_est_exp_mgcg_p = Path("salidas") / "Clasificaciones" / "CHECK_EST_EXP_MGCG_P.xlsb"
ruta_check_est_exp_mgcg_p.parent.mkdir(parents=True, exist_ok=True)
check_est_exp_mgcg_p.to_excel(ruta_check_est_exp_mgcg_p, sheet_name="PROPUESTA", index=False)


In [ ]:
# módulo 2: importa clasificaciones desde excel y las ingresa a la tabla principal
# IMPORTA CLASIFICACIONES (el bridge PC Files Server no aplica en Python: se lee directo por la ruta local)
est_exp_mgcg_p = pd.read_excel(ruta_check_est_exp_mgcg_p, sheet_name="PROPUESTA")
est_exp_mgcg_p.to_sql("#est_exp_mgcg_p", work_conn, if_exists="replace", index=False)


In [ ]:
work_conn.execute(text("DELETE FROM #est_exp_mgcg_p WHERE (C_CUENTA IS NULL OR C_CUENTA = '') AND (C_ITEM IS NULL OR C_ITEM = '')"))


In [ ]:
# VERIFICA TOTALES DE ESTRUCTURAS MGCG DEL PERIODO
work_conn.execute(text("DROP TABLE IF EXISTS #check_mgcg_tot"))
sql_check_mgcg_tot = """
SELECT MES, C_CUENTA, C_ITEM, SUM(POND) AS TOTAL, ROUND(SUM(POND), 1) AS TOTAL2
INTO #check_mgcg_tot
FROM #est_exp_mgcg_p
GROUP BY MES, C_CUENTA, C_ITEM
"""
work_conn.execute(text(sql_check_mgcg_tot))
work_conn.execute(text("DELETE FROM #check_mgcg_tot WHERE TOTAL2 = 1"))
check_mgcg_tot = pd.read_sql(text("SELECT * FROM #check_mgcg_tot"), work_conn)
_log("check_mgcg_tot", check_mgcg_tot)


In [ ]:
# ANEXA DATOS A TABLA PRINCIPAL: descarta lo que ya existe en GOBGENER.EST_EXP_MGCG antes de anexar
work_conn.execute(text("""
DELETE FROM #est_exp_mgcg_p
WHERE EXISTS (
    SELECT 1 FROM GOBGENER.dbo.EST_EXP_MGCG e
    WHERE e.C_CUENTA = #est_exp_mgcg_p.C_CUENTA AND e.C_ITEM = #est_exp_mgcg_p.C_ITEM
)
"""))


In [ ]:
est_exp_mgcg_p_final = pd.read_sql(text("SELECT * FROM #est_exp_mgcg_p"), work_conn)
est_exp_mgcg_p_final.to_sql("EST_EXP_MGCG", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.EST_EXP_MGCG", len(est_exp_mgcg_p_final))


In [ ]:
for t in ["#check_est_exp_mgcg", "#check_est_exp_mgcg_p", "#check_mgcg_tot", "#est_exp_mgcg_p"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## 7f-Check Est_Exp_MGG TCons_mod

Detecta cuentas-item de estructuras MGG sin clasificación, propone estructura para revisión manual vía Excel, y anexa las nuevas clasificaciones validadas a la tabla principal EST_EXP_MGG

*confianza: medium · verificador: approve · SAS: PROC SQL con LEFT JOIN + PROC EXPORT/IMPORT EXCELCS + PROC DATASETS APPEND*

In [ ]:
# ========= 7f-Check Est_Exp_MGG TCons_mod =========
# modificado junio 2025
# chequea consistencia tcons mgg
# módulo 1: determina faltantes y exporta datos a excel
# DETECTA CUENTAS-ITEM SIN ESTRUCTURAS
work_conn.execute(text("DROP TABLE IF EXISTS #check_est_exp_mgg"))
sql_check_est_exp_mgg = """
SELECT DISTINCT t1.C_CUENTA, t1.C_ITEM, COUNT(t1.C_ASIGNACION) AS ID
INTO #check_est_exp_mgg
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN GOBGENER.dbo.EST_EXP_MGG t2 ON t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
WHERE t1.Clasif_Tcons = 'MGG' AND t1.M_Muestra = 0 AND t2.C_SCN IS NULL
GROUP BY t1.C_CUENTA, t1.C_ITEM
"""
work_conn.execute(text(sql_check_est_exp_mgg))
check_est_exp_mgg = pd.read_sql(text("SELECT * FROM #check_est_exp_mgg"), work_conn)
_log("check_est_exp_mgg", check_est_exp_mgg)


In [ ]:
# Propone estructura si es que apareciera un item nuevo para la cuenta 24 clasificado en este tipo de transferencia consolidable
work_conn.execute(text("DROP TABLE IF EXISTS #check_est_exp_mgg_p"))
sql_check_est_exp_mgg_p = """
SELECT t1.C_CUENTA, t1.C_ITEM, t2.C_SCN, SUM(t2.POND) AS POND, t2.C_CAGENTE, t2.MES
INTO #check_est_exp_mgg_p
FROM #check_est_exp_mgg t1
LEFT JOIN GOBGENER.dbo.EST_EXP_MGG t2 ON t2.C_CUENTA = t1.C_CUENTA
WHERE t2.C_ITEM = '01'
GROUP BY t1.C_CUENTA, t1.C_ITEM, t2.C_SCN, t2.C_CAGENTE, t2.MES
"""
work_conn.execute(text(sql_check_est_exp_mgg_p))
check_est_exp_mgg_p = pd.read_sql(text("SELECT * FROM #check_est_exp_mgg_p"), work_conn)
_log("check_est_exp_mgg_p", check_est_exp_mgg_p)


In [ ]:
# módulo 1: exporta datos a excel (ruta relativa al workspace, reemplaza el UNC original)
ruta_check_mgg = Path("salidas") / "Clasificaciones" / "CHECK_EST_EXP_MGG_P.xlsb"
ruta_check_mgg.parent.mkdir(parents=True, exist_ok=True)
# PROC EXPORT DBMS=EXCELCS via bridge PC Files Server: se traduce como escritura directa a archivo
# pandas no soporta escritura de .xlsb; se exporta en formato xlsx equivalente
ruta_check_mgg_xlsx = ruta_check_mgg.with_suffix(".xlsx")
check_est_exp_mgg_p.to_excel(ruta_check_mgg_xlsx, sheet_name="PROPUESTA", index=False)


In [ ]:
# módulo 2: importa clasificaciones desde excel y las ingresa a la tabla principal
# IMPORTA CLASIFICACIONES (misma ruta relativa usada en el export; el bridge EXCELCS no existe en Python)
est_exp_mgg_p = pd.read_excel(ruta_check_mgg_xlsx, sheet_name="PROPUESTA")
est_exp_mgg_p.to_sql("#est_exp_mgg_p", work_conn, if_exists="replace", index=False)


In [ ]:
work_conn.execute(text("DELETE FROM #est_exp_mgg_p WHERE (C_CUENTA IS NULL OR C_CUENTA = '') AND (C_ITEM IS NULL OR C_ITEM = '')"))


In [ ]:
# VERIFICA TOTALES DE ESTRUCTURAS MGG DEL PERIODO
work_conn.execute(text("DROP TABLE IF EXISTS #check_mgg_tot"))
sql_check_mgg_tot = """
SELECT MES, C_CUENTA, C_ITEM, SUM(POND) AS TOTAL, ROUND(SUM(POND), 1) AS TOTAL2
INTO #check_mgg_tot
FROM #est_exp_mgg_p
GROUP BY MES, C_CUENTA, C_ITEM
"""
work_conn.execute(text(sql_check_mgg_tot))
work_conn.execute(text("DELETE FROM #check_mgg_tot WHERE TOTAL2 = 1"))
check_mgg_tot = pd.read_sql(text("SELECT * FROM #check_mgg_tot"), work_conn)
_log("check_mgg_tot", check_mgg_tot)


In [ ]:
# ANEXA DATOS A TABLA PRINCIPAL
work_conn.execute(text("""
DELETE FROM #est_exp_mgg_p
WHERE EXISTS (
    SELECT 1 FROM GOBGENER.dbo.EST_EXP_MGG e
    WHERE e.C_CUENTA = #est_exp_mgg_p.C_CUENTA AND e.C_ITEM = #est_exp_mgg_p.C_ITEM
)
"""))


In [ ]:
est_exp_mgg_p_final = pd.read_sql(text("SELECT * FROM #est_exp_mgg_p"), work_conn)
est_exp_mgg_p_final.to_sql("EST_EXP_MGG", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.EST_EXP_MGG", len(est_exp_mgg_p_final))


## 9a-Check Nvas Ent COFOG

Detecta entidades con ejecución del año que no tienen actividad COFOG clasificada, para su posterior imputación

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN y filtro IS NULL sobre AÑO*

In [ ]:
# ========= 9a-Check Nvas Ent COFOG =========
work_conn.execute(text("DROP TABLE IF EXISTS #check_cofog_entid"))
# año interpolado como entero (no :param) para que la #tmp sobreviva
# REVISA SI HAY ENTIDADES NUEVAS SIN ACTIVIDAD CLASIFICADA, EN LA CARGA
sql_check_cofog_entid = f"""
SELECT DISTINCT t1.C_ENTIDAD AS ENTIDAD,
       t2.C_COFOG,
       t2.N_COFOG,
       t2.C_COFOG_1,
       t2.N_COFOG_1,
       t1.C_PARTIDA,
       t1.N_PARTIDA,
       t1.C_CAPITULO,
       t1.N_CAPITULO,
       t1.C_PROGRAMA,
       t1.N_PROGRAMA
INTO #check_cofog_entid
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.COFOG t2 ON (t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_PARTIDA = t2.C_PARTIDA
    AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA)
WHERE (t2.C_COFOG IS NULL) AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_check_cofog_entid))


In [ ]:
check_cofog_entid = pd.read_sql(text("SELECT * FROM #check_cofog_entid"), work_conn)
_log("check_cofog_entid", check_cofog_entid)


## 9b-Clasif COFOG de Nvas Ent

Agrega a la matriz COFOG las clasificaciones imputadas para las nuevas entidades

*confianza: medium · verificador: approve · SAS: PROC DATASETS APPEND BASE=... DATA=... FORCE*

In [ ]:
# ========= 9b-Clasif COFOG de Nvas Ent =========
# ACTUALIZAR MATRIZ DE COFOG CON IMPUTACIONES PARA NUEVAS ENTIDADES
# PROC DATASETS APPEND: agrega check_cofog_entid (dejado en memoria por el nodo 9a) a GOBGENER.COFOG
work_conn.execute(text("DROP TABLE IF EXISTS #check_cofog_entid"))
check_cofog_entid.to_sql("#check_cofog_entid", work_conn, if_exists="replace", index=False)
sql_append = """
INSERT INTO GOBGENER.dbo.COFOG
SELECT * FROM #check_cofog_entid
"""
res = work_conn.execute(text(sql_append))
_log("APPEND GOBGENER.dbo.COFOG", res.rowcount)


## 10a-Check Analit COFOG

Detecta transacciones de cuentas 24/33 en entidades regionales del año sin actividad COFOG clasificada

*confianza: high · verificador: approve · SAS: PROC SQL CREATE TABLE con LEFT JOIN múltiple y filtro IS NULL*

In [ ]:
# ========= 10a-Check Analit COFOG =========
# REVISA SI HAY TRANSACCIONES NUEVAS DE GOREs SIN ACTIVIDAD CLASIFICADA, EN LA CARGA
sql_check_cofog_analit = """
SELECT DISTINCT t1.SUBSECTOR,
                t1.MONEDA,
                t1.C_TIPO_CUENTA,
                t1.C_CUENTA,
                t1.C_ITEM,
                t2.C_SCN,
                t1.C_ANALITICO,
                t1.N_ANALITICO,
                t2.C_CAGENTE,
                t3.C_COFOG_1,
                t3.N_COFOG_1,
                t1.C_PARTIDA,
                t1.N_PARTIDA,
                t1.C_CAPITULO,
                t1.N_CAPITULO,
                t1.C_PROGRAMA,
                t1.N_PROGRAMA
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.COFOG_T_SCN_2 t3 ON (t1.C_PARTIDA = t3.C_PARTIDA AND
    t1.C_CAPITULO = t3.C_CAPITULO AND t1.C_PROGRAMA = t3.C_PROGRAMA AND
    t1.C_CUENTA = t3.C_CUENTA AND t1.C_ANALITICO = t3.C_ANALITICO AND
    t1.MONEDA = t3.MONEDA)
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA AND
    t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION =
    t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO)
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4 ON (t1.C_ENTIDAD = t4.ENTIDAD)
WHERE (t3.C_COFOG_1 IS NULL) AND t1.AÑO = :year AND t1.C_CUENTA IN ('24','33')
AND t4.NIVEL = 'REGIONAL'
"""
check_cofog_analit = pd.read_sql(text(sql_check_cofog_analit), engine, params={"year": Year})
_log("check_cofog_analit", check_cofog_analit)


## 10b-Clasif Analit COFOG

Anexa a la matriz COFOG las imputaciones de nuevas entidades, renombrando N_ANALITICO a ANALITICO

*confianza: medium · verificador: approve · SAS: DATA step RENAME + PROC APPEND (FORCE)*

In [ ]:
# ========= 10b-Clasif Analit COFOG =========
work_conn.execute(text("DROP TABLE IF EXISTS #check_cofog_analit"))
sql_check_cofog_analit = """
SELECT *
INTO #check_cofog_analit
FROM #CHECK_COFOG_ANALIT
"""
work_conn.execute(text(sql_check_cofog_analit))


In [ ]:
# ACTUALIZAR MATRIZ DE COFOG CON IMPUTACIONES PARA NUEVAS ENTIDADES
# DATA STEP renombra N_ANALITICO a ANALITICO antes de anexar a la tabla destino
work_conn.execute(text("DROP TABLE IF EXISTS #check_cofog_analit_ren"))
sql_rename = """
SELECT *, N_ANALITICO AS ANALITICO
INTO #check_cofog_analit_ren
FROM #check_cofog_analit
"""
work_conn.execute(text(sql_rename))
work_conn.execute(text("ALTER TABLE #check_cofog_analit_ren DROP COLUMN N_ANALITICO"))
check_cofog_analit = pd.read_sql(text("SELECT * FROM #check_cofog_analit_ren"), work_conn)
_log("check_cofog_analit", check_cofog_analit)


In [ ]:
# PROC APPEND: agrega sin deduplicar, tal como el SAS original
check_cofog_analit.to_sql("COFOG_T_SCN_2", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.COFOG_T_SCN_2", len(check_cofog_analit))


## 8a-Check Nvas Ent ACTIV

Detecta entidades nuevas de EJECUCION sin ACTIVIDAD/NIVEL clasificado en CR18_APERTURA e imputa esos valores según el diccionario de partidas/capítulos

*confianza: high · verificador: approve · SAS: PROC SQL: LEFT JOIN de control (anti-join) + UPDATEs secuenciales de imputación por diccionario de partidas*

In [ ]:
# ========= 8a-Check Nvas Ent ACTIV =========
work_conn.execute(text("DROP TABLE IF EXISTS #check_activ_entid"))
# año interpolado como entero (no :param) para que la #tmp sobreviva
# REVISA SI HAY ENTIDADES NUEVAS SIN ACTIVIDAD CLASIFICADA, EN LA CARGA
sql_check_activ_entid = f"""
SELECT DISTINCT t1.C_ENTIDAD AS ENTIDAD,
       t2.ACTIVIDAD,
       t2.NIVEL,
       t1.C_PARTIDA,
       t1.N_PARTIDA,
       t1.C_CAPITULO,
       t1.N_CAPITULO,
       t1.C_PROGRAMA,
       t1.N_PROGRAMA
INTO #check_activ_entid
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t2 ON t1.C_ENTIDAD = t2.ENTIDAD
WHERE t2.ENTIDAD IS NULL AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_check_activ_entid))


In [ ]:
# PARTIDAS QUE SE CLASIFICAN COMPLETAMENTE EN AAPP-CENTRAL
res = work_conn.execute(text("""
UPDATE #check_activ_entid
SET ACTIVIDAD = 'AAPP', NIVEL = 'CENTRAL'
WHERE C_PARTIDA IN ('01','02','03','04','06','07','10','12','13','14','17','18','19','20','21','22','23','24','25','26','27','28','29')
"""))
_log("UPDATE #check_activ_entid AAPP-CENTRAL", res.rowcount)


In [ ]:
# EXCEPTO CUANDO CAPITULO ES CASA DE MONEDA
res = work_conn.execute(text("""
UPDATE #check_activ_entid
SET ACTIVIDAD = 'AAPP', NIVEL = 'CENTRAL'
WHERE C_PARTIDA = '08' AND C_CAPITULO NOT IN ('06') AND (ACTIVIDAD IS NULL OR ACTIVIDAD = '')
"""))
_log("UPDATE #check_activ_entid Partida 08", res.rowcount)


In [ ]:
# MINISTERIO DEL TRABAJO, CREDITO PRENDARIO
res = work_conn.execute(text("""
UPDATE #check_activ_entid
SET ACTIVIDAD = 'EXCLUIDO', NIVEL = 'EXCLUIDO'
WHERE C_PARTIDA = '15' AND C_CAPITULO = '04' AND (ACTIVIDAD IS NULL OR ACTIVIDAD = '')
"""))
_log("UPDATE #check_activ_entid Partida 15", res.rowcount)


In [ ]:
# MINISTERIO DE EDUCACIÓN, CAPITULO 11 JUNJI, SERV LOCALES DE EDUCACION: CAPITULO 18-19-21-22-23-24-25-26-27-28-29-30-31-32-33-34-35-36-37-38-39-40-41-42-43-44-45 AL 56
res = work_conn.execute(text("""
UPDATE #check_activ_entid
SET ACTIVIDAD = 'EEPP', NIVEL = 'CENTRAL'
WHERE (ACTIVIDAD IS NULL OR ACTIVIDAD = '') AND C_PARTIDA = '09' AND
    C_CAPITULO IN ('11','18','19','21','22','23','24','25','26','27','28','29','30','31','32','33','34','35','36','37','38','39','40','41','42','43','44','45','46','47','48','49','50','51','52','53','54','55','56','57','58','59','60','61','62','63','64','65','66')
"""))
_log("UPDATE #check_activ_entid Partida 09", res.rowcount)


In [ ]:
# PARTIDAS QUE SE CLASIFICAN COMPLETAMENTE EN AAPP-regional
res = work_conn.execute(text("""
UPDATE #check_activ_entid
SET ACTIVIDAD = 'AAPP', NIVEL = 'REGIONAL'
WHERE (ACTIVIDAD IS NULL OR ACTIVIDAD = '') AND (NIVEL IS NULL OR NIVEL = '') AND C_PARTIDA IN ('31')
"""))
_log("UPDATE #check_activ_entid Partida 31", res.rowcount)


In [ ]:
check_activ_entid = pd.read_sql(text("SELECT * FROM #check_activ_entid"), work_conn)
_log("check_activ_entid", check_activ_entid)


## 8b-Clasif ACTIV de Nvas Ent

Incorpora actividades de nuevas entidades a la matriz de apertura y a los directorios DIPRES (general y por año), reemplazando primero las combinaciones ya cubiertas

*confianza: medium · verificador: approve · SAS: PROC DATASETS APPEND + PROC SQL CREATE/DELETE/APPEND encadenados sobre tablas de BD y #tmp de sesión*

In [ ]:
# ========= 8b-Clasif ACTIV de Nvas Ent =========
# ACTUALIZAR MATRIZ DE ACTIVIDADES CON IMPUTACIONES PARA NUEVAS ENTIDADES
# APPEND server-side: CHECK_ACTIV_ENTID nace de una #tmp (session_sql)
cols_cr18_apertura = ", ".join(check_activ_entid.columns)
sql_append_apertura = f"""
INSERT INTO GOBGENER.dbo.CR18_APERTURA ({cols_cr18_apertura})
SELECT {cols_cr18_apertura}
FROM #check_activ_entid
"""
res = work_conn.execute(text(sql_append_apertura))
_log("APPEND GOBGENER.dbo.CR18_APERTURA", res.rowcount)


In [ ]:
# ACTUALIZAR MATRIZ PARALELA PARA EJECUCION DIPRES CON PACAPR
work_conn.execute(text("DROP TABLE IF EXISTS #cr18_apertura_pacapr_bd"))
sql_pacapr_bd = """
SELECT DISTINCT
       t1.ACTIVIDAD,
       t2.C_PARTIDA,
       t2.C_CAPITULO,
       t2.C_PROGRAMA,
       t1.NIVEL
INTO #cr18_apertura_pacapr_bd
FROM GOBGENER.dbo.CR18_APERTURA t1
LEFT JOIN GOBGENER.dbo.ENTIDADES t2 ON t1.ENTIDAD = t2.C_ENTIDAD
WHERE t2.C_PARTIDA IS NOT NULL
"""
work_conn.execute(text(sql_pacapr_bd))
cr18_apertura_pacapr_bd = pd.read_sql(text("SELECT * FROM #cr18_apertura_pacapr_bd"), work_conn)
_log("cr18_apertura_pacapr_bd", cr18_apertura_pacapr_bd)


In [ ]:
# Borra en CR18_ENTID_DIP las combinaciones ya cubiertas por la nueva apertura, antes del append
sql_delete_entid_dip = """
DELETE FROM GOBGENER.dbo.CR18_ENTID_DIP
WHERE EXISTS (
    SELECT 1 FROM #cr18_apertura_pacapr_bd p
    WHERE p.C_PARTIDA = CR18_ENTID_DIP.C_PARTIDA
      AND p.C_CAPITULO = CR18_ENTID_DIP.C_CAPITULO
      AND p.C_PROGRAMA = CR18_ENTID_DIP.C_PROGRAMA
)
"""
res = work_conn.execute(text(sql_delete_entid_dip))
_log("DELETE GOBGENER.dbo.CR18_ENTID_DIP", res.rowcount)


In [ ]:
# APPEND server-side de la matriz paralela hacia CR18_ENTID_DIP
cols_entid_dip = "ACTIVIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, NIVEL"
sql_append_entid_dip = f"""
INSERT INTO GOBGENER.dbo.CR18_ENTID_DIP ({cols_entid_dip})
SELECT {cols_entid_dip}
FROM #cr18_apertura_pacapr_bd
"""
res = work_conn.execute(text(sql_append_entid_dip))
_log("APPEND GOBGENER.dbo.CR18_ENTID_DIP", res.rowcount)


In [ ]:
# MODIFICADO JUNIO 2025, PARA INCORPORAR DATOS A DIRECTORIO DIPRES POR AÑO
work_conn.execute(text("DROP TABLE IF EXISTS #entid_cgr_year"))
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_entid_cgr_year = f"""
SELECT t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t2.ACTIVIDAD, t2.NIVEL, COUNT(t1.MES) AS ID
INTO #entid_cgr_year
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t2 ON t1.C_ENTIDAD = t2.ENTIDAD
WHERE t1.AÑO = {int(Year)}
GROUP BY t1.AÑO, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t2.ACTIVIDAD, t2.NIVEL
"""
work_conn.execute(text(sql_entid_cgr_year))
# DATA step original solo dropeaba la columna ID (era el COUNT auxiliar del GROUP BY)
entid_cgr_year = pd.read_sql(text("SELECT AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, ACTIVIDAD, NIVEL FROM #entid_cgr_year"), work_conn)
_log("entid_cgr_year", entid_cgr_year)


In [ ]:
# Reemplaza el año en CR18_ENTID_DIP_Y antes de cargar el nuevo detalle
sql_delete_entid_dip_y = "DELETE FROM GOBGENER.dbo.CR18_ENTID_DIP_Y WHERE AÑO = :year"
res = work_conn.execute(text(sql_delete_entid_dip_y), {"year": int(Year)})
_log("DELETE GOBGENER.dbo.CR18_ENTID_DIP_Y", res.rowcount)


In [ ]:
# APPEND server-side de ENTID_CGR_YEAR (sin la columna ID) hacia CR18_ENTID_DIP_Y
cols_entid_dip_y = "AÑO, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, ACTIVIDAD, NIVEL"
sql_append_entid_dip_y = f"""
INSERT INTO GOBGENER.dbo.CR18_ENTID_DIP_Y ({cols_entid_dip_y})
SELECT {cols_entid_dip_y}
FROM #entid_cgr_year
"""
res = work_conn.execute(text(sql_append_entid_dip_y))
_log("APPEND GOBGENER.dbo.CR18_ENTID_DIP_Y", res.rowcount)


In [ ]:
# El SAS dropeaba ENTID_CGR_YEAR al final; con la sesión se libera explícitamente la #tmp al terminar de usarla
work_conn.execute(text("DROP TABLE IF EXISTS #entid_cgr_year"))


## OPC 1a-Check Name Partida

Detecta entidades cuyo nombre de partida cambió, comparando fonéticamente el nombre antiguo contra el nuevo (umbral de acierto < 50%)

*confianza: medium · verificador: approve · SAS: DATA steps con SOUNDEX/SPEDIS + PROC SQL DISTINCT WHERE*

In [ ]:
# ========= OPC 1a-Check Name Partida =========
# COMPARA NOMBRE PARTIDA
compara_partida_1 = compara_pacapr.copy()
compara_partida_1["Partida_base"] = compara_partida_1["N_PARTIDA_ANTIG"].apply(_soundex)
compara_partida_1["Partida_nuevo"] = compara_partida_1["N_PARTIDA"].apply(_soundex)
_log("compara_partida_1", compara_partida_1)


In [ ]:
compara_partida_2 = compara_partida_1.copy()
compara_partida_2["Distance"] = compara_partida_2.apply(lambda r: _spedis(r["Partida_base"], r["Partida_nuevo"]), axis=1)
compara_partida_2["Porc_acierto"] = 100 - compara_partida_2["Distance"]
compara_partida_2.loc[compara_partida_2["Distance"] > 100, "Porc_acierto"] = 0
_log("compara_partida_2", compara_partida_2)


In [ ]:
# CREA TABLA CON ENTIDADES QUE CAMBIARON NOMBRE DE PARTIDA
compara_partida = compara_partida_2[compara_partida_2["Porc_acierto"] < 50].drop_duplicates()
_log("compara_partida", compara_partida)


## OPC 1b-Clasif Name Partida

Elimina de ENTIDADES las filas cuya partida/capítulo/programa/entidad cambiaron de nombre y las reincorpora con el nombre nuevo desde COMPARA_PARTIDA

*confianza: medium · verificador: approve · SAS: PROC SQL DELETE correlacionado + PROC DATASETS APPEND FORCE*

In [ ]:
# ========= OPC 1b-Clasif Name Partida =========
compara_partida.to_sql("#compara_partida", work_conn, if_exists="replace", index=False)


In [ ]:
# AGREGA TABLA CON ENTIDADES QUE CAMBIARON NOMBRE DE PARTIDA
# DELETE FROM GOBGENER.ENTIDADES WHERE EXISTS(...) — la referencia original 'ENTIDAD.C_ENTIDAD' era un typo
# (ENTIDAD es el libref, no la tabla del DELETE); se corrige a COMPARA_PARTIDA.C_ENTIDAD, que es la
# comparación que la deduplicación realmente necesita.
sql_delete = """
DELETE FROM GOBGENER.dbo.ENTIDADES
WHERE EXISTS (
    SELECT 1 FROM #compara_partida cp
    WHERE cp.C_PARTIDA = ENTIDADES.C_PARTIDA
      AND cp.C_CAPITULO = ENTIDADES.C_CAPITULO
      AND cp.C_PROGRAMA = ENTIDADES.C_PROGRAMA
      AND cp.C_ENTIDAD = ENTIDADES.C_ENTIDAD
)
"""
res = work_conn.execute(text(sql_delete))
_log("DELETE GOBGENER.dbo.ENTIDADES", res.rowcount)


In [ ]:
# PROC DATASETS; APPEND BASE=GOBGENER.ENTIDADES DATA=COMPARA_PARTIDA FORCE — acumula, tal cual el SAS
compara_partida.to_sql("ENTIDADES", engine, schema="dbo", if_exists="append", index=False)


## OPC 2a-Check Name Capitulo

Detecta entidades cuyo nombre de capítulo cambió, comparando similitud fonética/textual entre nombre antiguo y nuevo con umbral de acierto menor a 50%

*confianza: low · verificador: approve · SAS: DATA step con SOUNDEX/SPEDIS + PROC SQL SELECT DISTINCT con filtro*

In [ ]:
# ========= OPC 2a-Check Name Capitulo =========
# COMPARA NOMBRE CAPITULO
compara_capitulo_1 = compara_pacapr.copy()
compara_capitulo_1["Partida_base"] = compara_capitulo_1["N_CAPITULO_ANTIG"].apply(_soundex)
compara_capitulo_1["Partida_nuevo"] = compara_capitulo_1["N_CAPITULO"].apply(_soundex)
_log("compara_capitulo_1", compara_capitulo_1)


In [ ]:
compara_capitulo_2 = compara_capitulo_1.copy()
compara_capitulo_2["Distance"] = compara_capitulo_2.apply(
    lambda r: _spedis(r["Partida_base"], r["Partida_nuevo"]), axis=1
)
compara_capitulo_2["Porc_acierto"] = 100 - compara_capitulo_2["Distance"]
compara_capitulo_2.loc[compara_capitulo_2["Distance"] > 100, "Porc_acierto"] = 0
_log("compara_capitulo_2", compara_capitulo_2)


In [ ]:
# CREA TABLA CON ENTIDADES QUE CAMBIARON NOMBRE DE CAPITULO
compara_capitulo = compara_capitulo_2[compara_capitulo_2["Porc_acierto"] < 50].drop_duplicates()
_log("compara_capitulo", compara_capitulo)


## OPC 2b-Clasif Name Capitulo

Reemplaza en el maestro de entidades las filas cuyo capítulo cambió de nombre, agregando la versión actualizada

*confianza: medium · verificador: approve · SAS: PROC SQL DELETE correlacionado + PROC DATASETS APPEND*

In [ ]:
# ========= OPC 2b-Clasif Name Capitulo =========
# AGREGA TABLA CON ENTIDADES QUE CAMBIARON NOMBRE DE CAPITULO
# Elimina de GOBGENER.ENTIDADES las filas cuya combinación partida/capitulo/programa/entidad
# coincide con la tabla temporal #compara_capitulo (subida desde el WORK ya calculado)
work_conn.execute(text("DROP TABLE IF EXISTS #compara_capitulo"))
compara_capitulo.to_sql("#compara_capitulo", work_conn, if_exists="replace", index=False)
res = work_conn.execute(text("""
DELETE FROM GOBGENER.dbo.ENTIDADES
WHERE EXISTS (
    SELECT 1 FROM #compara_capitulo cc
    WHERE cc.C_PARTIDA = ENTIDADES.C_PARTIDA
      AND cc.C_CAPITULO = ENTIDADES.C_CAPITULO
      AND cc.C_PROGRAMA = ENTIDADES.C_PROGRAMA
      AND cc.C_ENTIDAD = ENTIDADES.C_ENTIDAD
)
"""))
_log("DELETE GOBGENER.dbo.ENTIDADES", res.rowcount)


In [ ]:
# PROC DATASETS; APPEND BASE=GOBGENER.ENTIDADES DATA=COMPARA_CAPITULO FORCE;
compara_capitulo.to_sql("ENTIDADES", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.ENTIDADES", len(compara_capitulo))


## OPC 3a-Check Name Programa

Identifica entidades cuyo nombre de programa cambió, comparando nombre antiguo vs nuevo con similitud fonética/tipográfica bajo el 50% de acierto

*confianza: high · verificador: approve · SAS: DATA steps con SOUNDEX/SPEDIS + PROC SQL SELECT DISTINCT con filtro*

In [ ]:
# ========= OPC 3a-Check Name Programa =========
# COMPARA NOMBRE PROGRAMA
compara_programa_1 = compara_pacapr.copy()
compara_programa_1["Partida_base"] = compara_programa_1["N_PROGRAMA_ANTIG"].apply(_soundex)
compara_programa_1["Partida_nuevo"] = compara_programa_1["N_PROGRAMA"].apply(_soundex)
_log("compara_programa_1", compara_programa_1)


In [ ]:
compara_programa_2 = compara_programa_1.copy()
compara_programa_2["Distance"] = compara_programa_2.apply(
    lambda r: _spedis(r["Programa_base"], r["Programa_nuevo"]), axis=1
)
compara_programa_2["Porc_acierto"] = 100 - compara_programa_2["Distance"]
compara_programa_2.loc[compara_programa_2["Distance"] > 100, "Porc_acierto"] = 0
_log("compara_programa_2", compara_programa_2)


In [ ]:
# CREA TABLA CON ENTIDADES QUE CAMBIARON NOMBRE DE PROGRAMA
compara_programa = compara_programa_2[compara_programa_2["Porc_acierto"] < 50].drop_duplicates()
_log("compara_programa", compara_programa)


## OPC 3b-Clasif Name Programa

Reemplaza en Entidades los registros de partida/capítulo/programa/entidad que cambiaron de nombre de programa, cargando las versiones actualizadas

*confianza: medium · verificador: approve · SAS: PROC SQL DELETE correlacionado + PROC DATASETS APPEND*

In [ ]:
# ========= OPC 3b-Clasif Name Programa =========
# AGREGA TABLA CON ENTIDADES QUE CAMBIARON NOMBRE DE PROGRAMA
work_conn.execute(
    text("DROP TABLE IF EXISTS #compara_programa")
)
compara_programa.to_sql("#compara_programa", work_conn, if_exists="replace", index=False)


In [ ]:
# DELETE FROM GOBGENER.ENTIDADES WHERE EXISTS(...) -- el SAS original comparaba C_ENTIDAD contra el libref ENTIDAD (typo),
# que aqui equivale a la propia tabla COMPARA_PROGRAMA (mismo alcance logico que ENTIDADES.C_ENTIDAD)
sql_delete = """
DELETE FROM GOBGENER.dbo.ENTIDADES
WHERE EXISTS (
    SELECT 1 FROM #compara_programa
    WHERE C_PARTIDA = ENTIDADES.C_PARTIDA
      AND C_CAPITULO = ENTIDADES.C_CAPITULO
      AND C_PROGRAMA = ENTIDADES.C_PROGRAMA
      AND C_ENTIDAD = ENTIDADES.C_ENTIDAD
)
"""
res = work_conn.execute(text(sql_delete))
_log("DELETE GOBGENER.dbo.ENTIDADES", res.rowcount)


In [ ]:
# PROC DATASETS; APPEND BASE=GOBGENER.ENTIDADES DATA=COMPARA_PROGRAMA FORCE
compara_programa.to_sql("ENTIDADES", engine, schema="dbo", if_exists="append", index=False)
_log("APPEND GOBGENER.dbo.ENTIDADES", len(compara_programa))
